# <center>HermesAnalytics 第二节课：从一次问数到连续分析与受控复核</center>

&emsp;&emsp;HermesAnalytics 第一节课已经完成了一件事：让一条可信问数主链路真实跑通。我们从安装进入系统，认识了四个服务、两个工作台、人工确认和证据链，最终证明“一句中文问题可以变成有来源、可复核的真实结果”。

&emsp;&emsp;第二节课要回答一个新问题：当分析师不是只问一次，而是连续追问、改口径、重置会话、进入 SQL 工作台检查参数、再回到结果并证明一个数字时，系统怎样保证每一步仍然可信？我们把这种能力概括为**连续分析与受控复核**。

&emsp;&emsp;学习方法延续第一节课：先看界面和状态，再回源码找责任边界。本节正文仍以界面与状态判断为主，每章末尾用一张「机制卡」下钻 nl2sql 规划侧一条确定性链（可离线运行，不依赖起服务）；不逐行读 SQLGlot AST、不展开执行侧与模型阶段，只为证明「模型提交结构化 JSON，SQL 由确定性程序生成，校验、冻结、证据全在模型循环之外」这条铁律的代码形态。

&emsp;&emsp;先认识四个词：**Intent** 是本轮完整分析意图；**Core** 在课件中指教学包装 `DemoPlanningCore`；生产由 `ClarificationDetector`、`PlanningToolBridge`、`Planner` / `Compiler` / `Policy` / `Freeze` 分责；**gaps** 是 Core 返回的缺口列表；**execution** 是一次与有效校验和实际参数绑定的受控执行记录。

> **📌 目标受众与前置要求**：你已经完成第一节课，了解 Agent、结构化 JSON、Tool Calling、SQL、关系型数据库和 HTTP API 基础，也知道人工确认、FrozenQuery、`plan_hash`、ResultSnapshot 和 FactRef 的基本概念。本课只补充专用 conda 环境，不重复 Docker 部署、总体架构和基础问数主链。

> **📌 学完本节你将带走七项能力**：① **澄清模糊问题**：依据 `gaps` 补齐指标、时间或 Top N；② **判断语义边界**：区分信息不完整、业务组合不支持与系统故障；③ **管理连续上下文**：判断继承、追加、切换和重置；④ **区分 SQL 模式**：依据来源与血缘判断返回资格；⑤ **复述受控复核**：能够按验收卡判断参数变化、重新校验、新 execution 与回流；⑥ **核对实际口径**：判断最终解释是否跟随实际执行参数；⑦ **判断证据定位**：能够按验收卡复述 FactRef 定位并分层诊断失败。

> **💡 学完本节不能做**：不能独立完成归因分析（归因工作台在后续课程展开）；不能修改数据库 schema 或管理领域包（数据库管理员变更在后续课程展开）；不能直接编写自由 SQL 并返回 Hermes 分析工作台（自由 SQL 结果只能留在工作台）。

> **📅 时效性说明**：本课唯一授课源码是 `HermesAnalytics第二版`。源码事实与相对链接均指向该版本；运行验证基于 Compose 项目 `hermes-analytics-final`，已覆盖：澄清对话、重置后历史保留、待确认 Query、可编辑 Seed、参数修改、VALID、SUCCEEDED、受控回流与 FactRef 点击定位。解释失败样例尚未取得运行截图，对应章节以机制说明为主。机制图表达组件关系，不充当运行截图。

&emsp;&emsp;下面用一张路线表固定全课顺序和唯一主任务。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-1　课程路线与唯一主任务</font></p>

<div class="center">

| 章节 | 这一段要回答的问题 | 主任务位置 |
| :---: | :---: | :---: |
| 课程承接 | 从“一次问数”到“能持续分析”缺什么？ | 固定任务与路线 |
| 第二章：Whole Task、确定性主链与 Agent 调用 | 连续分析由哪些验收节点组成？确定性主链怎么亲手跑通？ | 建立五步判断地图 |
| 第二章：Agent 调用（2.1-2.3） | Agent 怎么准备、怎么防越界、裸调用会发生什么？ | 模型侧入口 |
| 第三章：领域包与 Agent 完整调用 | 模型凭什么选对表字段？提交的 Intent 怎样进入确定性主链？ | 语义地基 + 全链接通 |
| 第四章：澄清与语义边界 + 机制卡① | 系统反问、拒绝、排障分别说明什么？ | 形成完整分析意图 |
| 第五章：会话与上下文 + 机制卡② | 哪些条件保留、替换、丢弃、清空？ | 管理多轮变化 |
| 第六章：SQL 工作台受控复核 + 机制卡③ | 为什么自由 SQL 不能返回 Hermes？ | 走通受控复核闭环 |
| 第七章：结果与证据诊断 + 机制卡④ | 怎样证明一个数字来自真实执行？ | 定位 FactRef 并分层诊断 |
| 第八章：迁移任务与总结 | 我能否独立完成连续分析任务？ | 完成全课验收 |

</div>

&emsp;&emsp;唯一主任务始终是下面这条连续会话。全课不增加第二主案例；后续章节会拆开其中每一步，但不会换成其他业务问题。

> **唯一主任务**：
> 1. 提交模糊问题“哪个渠道表现最好？”；
> 2. 补齐指标“净销售额”与时间“2026 年 6 月”；
> 3. 追问“只看华东地区呢？”；
> 4. 追问“支付订单数呢？”；
> 5. 执行“重新分析”；
> 6. 提交新问题“2026 年 6 月华东地区各渠道净销售额”；
> 7. 判断待确认 Query 是否具备联动入口资格；
> 8. 按验收卡判断批准参数变化后是否重新校验并产生新 execution；
> 9. 判断回流解释是否采用实际执行口径；
> 10. 按证据定位卡核对 FactRef 对应的行、列和值。

<p style="text-align: center !important; width: 100%;"><font face="黑体" size=4>开课前环境准备：注册 hermes-lesson2 Notebook kernel</font></p>

&emsp;&emsp;首次安装时，请在终端按顺序执行（如果环境已存在，可跳过第一条创建命令）：

```bash
conda create -n hermes-lesson2 python=3.12.13 -y
conda activate hermes-lesson2
python -m pip install -r requirements.txt
python -m pip check
python -m ipykernel install --user --name hermes-lesson2 --display-name hermes-lesson2
```

&emsp;&emsp;然后在 Notebook 的 Kernel 菜单中选择 `hermes-lesson2`。本课只安装 `requirements.txt` 声明的直接依赖；三个本地项目的传递依赖由各自的 `pyproject.toml` 与 `uv.lock` 管理，不复制 `pip freeze`。安装完成后，`python -m pip check` 应无依赖冲突；其中 `requests==2.33.0` 与 `websockets==15.0.1` 来自本地 `hermes-agent` 的源码锁定，不在本文件重复维护。不需要也不要在课件中输出任何密钥。

In [ ]:
!pip install -r requirements.txt -q


## <center>第一章：课程承接——从一次问数到能持续分析</center>

&emsp;&emsp;第一节课我们完成了一件事：提交一句话，系统在人工确认后执行，并给出结果、结论和证据。到了真实的经营分析场景，分析很少只问一次。分析师会先问一个模糊问题，再逐步补口径；会基于上一轮结果继续追问；会在换指标、换地区、换对象时判断哪些条件应该保留；也会在 SQL 工作台里复核参数，最后还要证明结论里的数字来自哪次真实执行。

&emsp;&emsp;所以本课的前置不是“重新部署”，而是把第一节课已经认识的能力重新组合成一条连续链路。四个关键前置需要记住：人工确认是独立暂停点；分析工作台和 SQL 工作台各自有入口资格；自由 SQL 永远不能返回 Hermes；证据必须来自服务端，而不是模型文字。

&emsp;&emsp;本课边界：我们只深入数据分析师这条主线，归因工作台和管理员变更在后续课程展开。下面这张表把第一节已完成与本课新增内容分开。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-2　第一节到第二节的边界</font></p>

<div class="center">

| 第一节已覆盖 | 第二节处理方式 |
| :---: | :---: |
| Docker、Compose 与四服务 | 不重复讲，只作为已掌握前置 |
| readiness、模型可用与 E2E 的区别 | 快速回顾，不重新演示 |
| 系统黑盒架构与五类职责 | 用前置能力卡恢复，不重新拆服务 |
| 5 个指标与 4 类维度 | 能力卡回顾 |
| 一次问数六阶段 | 状态图回顾 |
| FrozenQuery、`plan_hash` 与人工确认 | 作为联动 SQL 的入口前置，不深挖源码 |
| 分析工作台与 SQL 工作台基本区别 | 本课深入两种 SQL 的来源和返回资格 |
| 模型不能直接写并执行 SQL | 作为已知结论使用 |
| ResultSnapshot 与 FactRef 基本概念 | 本课升级为点击定位和失败诊断能力 |

</div>

&emsp;&emsp;本课新增内容：多轮澄清；上下文继承、追加、切换和重置；指标 × 维度业务语义边界；自由 SQL 与会话联动 SQL 的完整使用边界；参数契约和旧校验失效；工作台执行结果受控回流；实际执行口径覆盖原问题措辞；FactRef 定位；查询失败与结果解读失败的分层判断。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-3　本节数据分析师功能观察清单</font></p>

<div class="center">

| 功能 | 页面动作 | 关键状态 | 源码责任入口 | 一句话要点 |
| :---: | :---: | :---: | :---: | :---: |
| 连续澄清 | 模型按 `gaps`（缺口清单）提问，学员补齐指标/时间/Top N | `INTENT_INCOMPLETE`（意图不完整）、`gaps` | `ClarificationDetector.find_gaps` | 系统把缺什么说清楚，不替你猜 |
| 上下文继承/替换/重置 | 追加华东、换指标、“重新分析” | `current_context.last_intent` / `last_query_id` | `AnalystConversationWorkspace.tsx` | 每轮重新形成完整 Intent，不是自动拼 JSON |
| 人工确认点 | 冻结 Query 停在确认页 | `AWAITING_EXECUTION_CONFIRMATION` | `PlanningToolBridge._finish_if_ready`、`park_ready_for_confirmation`；页面展示见 `SqlExecutionConfirmationCard` | 执行前必须有人确认 |
| 会话联动 SQL | 从确认点进入 SQL 工作台 | Seed 未过期、来源生效 | `workbench_seed` | 只有待确认的冻结 Query 能带 Seed |
| 参数校验与执行 | 修改批准参数→重新校验→执行 | `VALID/ALLOW`、`SUCCEEDED` | `WorkbenchService.validate/execute` | 改参数后旧校验失效 |
| 受控回流 | 返回分析工作区 | `WORKBENCH_REVISION`、`ANALYSIS_VERIFIED` | `resume_from_workbench` | 只提交执行引用，不传结果 |
| 技术详情 | 查看执行状态与实际参数 | Query、execution、Snapshot | `AnalystTechnicalDetail`，`frontend/src/features/analysis/components/AnalystTechnicalDetail.tsx` | 先看执行到哪一步 |
| FactRef 定位 | 点击结论数字高亮行列值 | `allowed_fact_ref_ids` | `EvidenceBuilder.build` | 数字必须来自服务端坐标 |

</div>

&emsp;&emsp;**本章自测**：问题：为什么“已经会问 2026 年 6 月各渠道净销售额”仍可能不会处理“只看华东呢”？参考答案：因为“只看华东”不是新问题，而是对上一轮完整分析意图的继承性修改；需要判断哪些条件保留、哪些条件替换，并重新形成本轮完整口径。通过标准：能说出本课新增的七项能力，并指出第一节哪些内容不会再重复讲。

## <center>第二章：Whole Task、确定性主链与 Agent 调用</center>

&emsp;&emsp;在拆机制之前，我们先建立一张机制地图：从模糊问题开始，经过澄清、追问、重置、新建查询、联动 SQL、受控执行、回流，最后通过 FactRef 证明一个数字。这张地图是机制示意（源码核验），不是运行截图，用来固定五步全课骨架。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172938127.png" alt="Whole Task 五步主线机制示意图" width="85%"></div>

&emsp;&emsp;五步动作是补全意图 → 管理上下文 → 进入受控复核 → 按实际参数返回 → 定位证据。下面这张地图换成五步主线后，后续每章都把这些动作放回同一条连续会话，不增加第二案例。

> **Whole Task 证据状态**：已验证项包括澄清、重置、待确认、可编辑 Seed、改参、VALID、SUCCEEDED、受控回流和 FactRef 点击定位；结果解读失败界面尚未取得运行截图。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-4　Whole Task 五步主线与对应章节</font></p>

<div class="center">

| 动作 | 验收时应观察的结果 | 对应章节 |
| :---: | :---: | :---: |
| 补全意图 | 从“哪个渠道表现最好”到“2026 年 6 月各渠道净销售额” | 第四章 |
| 管理上下文 | 只看华东、支付订单数、重新分析 | 第五章 |
| 进入受控复核 | 从待确认 Query 进入会话联动 SQL | 第六章 |
| 按实际参数返回 | 修改批准参数、重新校验、执行、回流 | 第六章 |
| 定位证据 | 数字能够对应 FactRef 与同次 ResultSnapshot | 第七章 |

</div>

&emsp;&emsp;**验收卡**：五个动作的顺序是补全意图 → 管理上下文 → 进入受控复核 → 按实际参数返回 → 定位证据。你需要在后续每章结束后，把对应判断放回这张机制地图。

&emsp;&emsp;在进入后续章节拆机制之前，先亲手把「确定性主链」跑通一次：看一条结构化 Intent 怎样在没有模型参与的情况下，产出一条参数化 SQL 和一个 `plan_hash`。这是全课四张机制卡都要回到的一条线——`Intent → QueryPlan → SQL → PolicyResult → FrozenQuery`。

&emsp;&emsp;这里有两条明确分开的执行路径：普通 Notebook Code Cell 使用 `hermes-lesson2` kernel；下一格命令则显式调用仓库 `backend/.venv`。本机实测时，`.venv` 的运行时 editable 映射解析到当前 `HermesAnalytics第二版` 源码；个别 `direct_url.json` 仍保留历史目录文本，因此不能只靠安装元数据判断真实导入路径。命令继续显式设置 `PYTHONPATH="src:../database"`，把本次脚本使用的后端源码与数据库包路径写清楚，避免依赖启动目录或隐藏的 kernel 配置。

In [2]:
# 命令行操作：ipynb 单元格中以 ! 前缀直接运行（在终端执行时去掉 ! 即可）
# 使用本课件同级目录下的 HermesAnalytics第二版 仓库（无连字符；含 .env 与最新源码）
!cd HermesAnalytics第二版/backend && PYTHONPATH="src:../database" .venv/bin/python mainline.py

SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_orders FROM analytics.v_paid_order_items WHERE paid_at >= :period_start AND paid_at < :period_end GROUP BY channel_id, channel_name ORDER BY channel_id
{'period_start': datetime.date(2026, 6, 1), 'period_end': datetime.date(2026, 7, 1)}
True ['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS']
sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83


&emsp;&emsp;这条命令必须原样理解：`cd HermesAnalytics第二版/backend` 先进入后端目录；`PYTHONPATH="src:../database"` 临时补上后端源码与数据库包的导入路径；`.venv/bin/python` 明确使用仓库由 uv 管理的开发解释器；`mainline.py` 是仓库已提供、用于拆解确定性主链的脚本。conda 是本 Notebook 的 kernel，`.venv` 是仓库 uv 的开发解释器，两者不混用。本机两条路径均已在 P0 跑出 SQL、参数、Policy 与 `plan_hash`。

&emsp;&emsp;源码已提供，下面拆解同一脚本。它位于 `backend/` 目录，只调用四个确定性模块，不连数据库、不调模型。

In [3]:
# nl2sql 规划侧完整主链：Intent → QueryPlan → SQL → 白名单 → 冻结（全程无模型、无数据库）
from datetime import date
from pathlib import Path
from hermes_analytics.nl2sql.catalog.loader import DomainPackageLoader        # 读领域包 manifest.json
from hermes_analytics.nl2sql.intent.models import AnalysisIntent, TimeRange   # 七字段意图
from hermes_analytics.nl2sql.planning.planner import QueryPlanner            # ① 规划
from hermes_analytics.nl2sql.compilation.compiler import SqlCompiler         # ② 确定性编译
from hermes_analytics.nl2sql.policy.inspector import SqlPolicy               # ③ SQLGlot 白名单
from hermes_analytics.nl2sql.compilation.freeze import FrozenQueryFactory    # ④ plan_hash 冻结

cwd = Path.cwd()               # 兼容两种起跑位置：课件目录，或 backend/（上一级即仓库根）
ROOT = cwd / "HermesAnalytics第二版" if (cwd / "HermesAnalytics第二版").is_dir() else cwd.parent
catalog = DomainPackageLoader().load_file(ROOT / "database/domain/default-ecommerce/manifest.json")
intent = AnalysisIntent(
    metric="paid_orders",                                     # 指标：支付订单数
    time_range=TimeRange(start=date(2026, 6, 1), end=date(2026, 6, 30)),
    time_grain=None, group_by=("channel",), filters=(),
    comparison="none", ranking=None,
)
plan = QueryPlanner().plan(intent, catalog)                           # ① 意图 → 可重放计划
compiled = SqlCompiler().compile(plan, catalog, catalog.content_hash)  # ② 拼参数化 SQL
policy = SqlPolicy().inspect(compiled.sql, plan, catalog)             # ③ 四项安全判定
frozen = FrozenQueryFactory().freeze(compiled, plan, catalog)         # ④ plan_hash + 四份哈希

print(compiled.sql)                                    # 值只用命名占位符，不进 SQL 文本
print(compiled.parameters)                             # period_start/period_end 半开区间
print(policy.allowed, [d.rule_id for d in policy.decisions])  # 四项白名单全过
print(frozen.plan_hash)                                # sha256 前缀，内容寻址防篡改


SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_orders FROM analytics.v_paid_order_items WHERE paid_at >= :period_start AND paid_at < :period_end GROUP BY channel_id, channel_name ORDER BY channel_id
{'period_start': datetime.date(2026, 6, 1), 'period_end': datetime.date(2026, 7, 1)}
True ['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS']
sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83


&emsp;&emsp;运行输出示例：

```text
SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_orders
FROM analytics.v_paid_order_items
WHERE paid_at >= :period_start AND paid_at < :period_end
GROUP BY channel_id, channel_name ORDER BY channel_id

参数: {'period_start': 2026-06-01, 'period_end': 2026-07-01}   ← 半开区间，end = 结束日 + 1
白名单: True  [SQLP-001-SINGLE-READ, SQLP-002-GOVERNED-OBJECTS, SQLP-003-NO-STAR, SQLP-004-FUNCTIONS]
plan_hash: sha256:d9e3342d...   ← 规范化 SQL + 参数 + 计划 + 目录哈希 + 三份语义哈希
```

&emsp;&emsp;先记住这条主链的形状，后续章节的机制卡会逐个回到它的四个环节。如果跑不出这段，检查 `PYTHONPATH` 是否同时含 `src` 与 `../database`。

**Agent 调用：让模型提交结构化 Intent**

&emsp;&emsp;上面的 `mainline.py` 是一条**离线确定性链**——我们手动构造了 `AnalysisIntent` 对象，然后交由编译器、策略和冻结程序依次处理。但真实问数流程中，`AnalysisIntent` 不是人手工构造的，而是**模型通过 Hermes Agent 调用 `submit_query_plan` 工具提交的**。

&emsp;&emsp;下面我们补上链条的前半段：用 DeepSeek 模型跑 Hermes Agent -> 模型提交结构化查询计划 -> Core 收到提交后进入确定性主链。全程不连数据库、不执行 SQL。Agent 调用需要真实模型凭据——直接从 `HermesAnalytics第二版` 仓库根目录的 `.env` 读取（见下方环境准备块），确定性主链是离线运行的。

&emsp;&emsp;先看这条链路的两段分工：

```text
+-------------------------------------------------+
|  模型阶段（Agent 调用，凭据来自 .env）             |
|  用户问题 -> 系统提示词 + 领域包投影               |
|           -> 模型调 submit_query_plan 工具         |
|           -> 提交结构化 Intent（JSON）              |
+------------------------+------------------------+
                         | 结构化 Intent
                         v
+-------------------------------------------------+
|  确定性阶段（离线运行，不调模型）                   |
|  Intent -> QueryPlan -> SQL -> Policy -> FrozenQuery |
|  ^ 这就是 mainline.py 已经跑通的那条链            |
+-------------------------------------------------+
```

&emsp;&emsp;下面把整个过程拆成代码块，逐块执行、逐块观察输出。模型凭据不再手工 `export`——直接从 `HermesAnalytics第二版` 仓库根目录的 `.env` 读取四个模型变量（`HERMES_ANALYTICS_MODEL_PROVIDER` / `_NAME` / `_API_KEY` / `_BASE_URL`），凭据只进内存、全程不打印密钥。先进入 `backend/` 目录并加载凭据：

In [4]:
# 环境准备：进入 backend/ 目录 + 从仓库 .env 加载模型凭据（不打印密钥）
import os
from pathlib import Path

cwd = Path.cwd()                       # 兼容两种起跑位置：课件目录，或 cell 3 已 cd 进的 backend/
REPO = cwd / "HermesAnalytics第二版" if (cwd / "HermesAnalytics第二版").is_dir() else cwd.parent
os.chdir(REPO / "backend")   # 进入后端目录，后续代码块的 Path.cwd().parent 即仓库根

def load_env(path):
    """读取课程模型配置。

    Args:
        path: `.env` 文件路径。

    Returns:
        配置键值字典；文件缺失时返回空字典。禁止打印完整字典或密钥值。
    """
    env = {}
    try:
        lines = open(path, encoding="utf-8").read().splitlines()
    except FileNotFoundError:
        print(f"[WARN] 未找到 {path}——请先参照 .env.example 配置 .env")
        return env
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
    return env

ENV = load_env(REPO / ".env")
api_key = ENV.get("HERMES_ANALYTICS_MODEL_API_KEY")             # 密钥只进内存，不打印
provider = ENV.get("HERMES_ANALYTICS_MODEL_PROVIDER", "deepseek")
model_name = ENV.get("HERMES_ANALYTICS_MODEL_NAME", "deepseek-chat")
base_url = ENV.get("HERMES_ANALYTICS_MODEL_BASE_URL", "https://api.deepseek.com/v1")
print(f"[OK] 模型凭据已加载：provider={provider}, model={model_name}（密钥不打印）")


[OK] 模型凭据已加载：provider=openrouter, model=deepseek/deepseek-v4-flash（密钥不打印）


```text
# provider/model 随 .env 实际取值显示；当前仓库 .env 配置如下
[OK] 模型凭据已加载：provider=openrouter, model=deepseek/deepseek-v4-flash（密钥不打印）
```

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172950763.png" alt="模型提交结构化 Intent，确定性程序接管执行" width="100%"></div>
&emsp;&emsp;以下代码块按顺序执行，每个块的变量会延续到下一个块（和 Jupyter Notebook 的 cell 执行模型一致）。

### 2.1 准备阶段：领域包、提示词与 Agent 工厂

&emsp;&emsp;在调用 Agent 之前，需要先准备好三样东西：领域包、系统提示词、Agent 工厂。这三样不是各自独立的——**它们共同定义了 Agent 的能力边界**。领域包决定了模型能问哪些指标、按哪些维度分组、用哪些筛选算子；系统提示词把领域包中的业务知识投影成模型能读的 JSON；Agent 工厂创建锁死工具集的 Agent 实例，而工具的参数校验同样依赖领域包做确定性约束。换句话说：**换一份领域包，Agent 能问的问题就完全不同**。这三个步骤不涉及任何模型调用，全部是本地确定性操作。

&emsp;&emsp;**加载领域包。** 这份 `manifest.json` 是 Agent 能力的根基——模型不读数据库、不做 RAG，它只知道这份登记里写了什么。`DomainPackageLoader.load_file()` 读取并校验 JSON，返回 `DomainCatalog` 对象，包含 `metrics`（5 个指标定义）、`view`（治理视图查询方法）、`dimension`（维度查询方法）等。后面组装提示词、校验工具参数、读取维度取值，全都从这个 `catalog` 对象里取数据。

In [5]:
# 加载领域包
from pathlib import Path
from hermes_analytics.nl2sql.catalog.loader import DomainPackageLoader

ROOT = Path.cwd().parent  # backend/ 的上一级即仓库根
catalog = DomainPackageLoader().load_file(
    ROOT / "database/domain/default-ecommerce/manifest.json"  # 返回 DomainCatalog，含 metrics/views/dimensions/joins
)
print(f"[OK] 领域包已加载：{len(catalog.metrics)} 个指标")
for m in catalog.metrics:
    print(f"   - {m.id}（{m.display_name}）：{m.description[:40]}...")


[OK] 领域包已加载：5 个指标
   - net_sales（净销售额）：分析期内实付商品金额减去按退款发生时间归属的退款商品金额。...
   - paid_orders（支付订单数）：分析期内已支付订单 ID 的去重数。...
   - paid_buyers（支付买家数）：分析期内已支付订单 customer_key 的去重数。...
   - avg_order_value（客单价）：分析期内实付商品金额除以支付订单数。...
   - refund_amount_rate（退款金额率）：分析期内按退款发生时间归属的退款商品金额除以同期实付商品金额。...


```text
[OK] 领域包已加载：5 个指标
   - net_sales（净销售额）：分析期内实付商品金额减去按退款发生时间...
   - paid_orders（支付订单数）：分析期内已支付订单 ID 的去重数...
   - paid_buyers（支付买家数）：分析期内已支付订单 customer_key 的去重数...
   - avg_order_value（客单价）：分析期内实付商品金额除以支付订单数...
   - refund_amount_rate（退款金额率）：分析期内按退款发生时间归属的退款...
```

&emsp;&emsp;**组装系统提示词。** 系统提示词由两部分拼接而成，都依赖上面加载的 `catalog`：`core_prompt` 是规划阶段的能力模板（包含 Intent 七字段定义、工具使用规则、提交前自检清单），其中的 `<<槽位>>` 由能力清单填充——可用时间粒度、对比方式、排名字段、筛选算子、分组上限等全部来自领域包；`domain_addendum` 是把 `catalog` 中的 5 个指标定义、4 个维度取值列表投影成的精简 JSON，模型靠这份 JSON "知道"能问什么。生产环境中还有第三层 `turn_context`（服务端注入的 today、data_updated_at 等），演示时省略。

In [6]:
# 组装系统提示词
import json
from hermes_analytics.hermes_adapter.prompt import _CORE_PROMPTS
from hermes_analytics.application.analysis_planning.domain_content import (
    build_planning_domain_content,
)

planning_prompt = _CORE_PROMPTS["planning"]  # 规划阶段系统提示词模板，含 Intent 七字段定义、工具规则、自检清单
domain_addendum = json.dumps(
    build_planning_domain_content(catalog),  # 把 manifest 中的指标/维度/取值投影成精简 JSON
    ensure_ascii=False, indent=2
)
system_prompt = planning_prompt + "\n\n【领域】\n" + domain_addendum  # 拼接两层（生产环境由 PromptComposer 装配三层）
print(f"[OK] 系统提示词已组装：{len(system_prompt)} 字符")
print(f"   core_prompt 长度：{len(planning_prompt)} 字符")
print(f"   domain_addendum 长度：{len(domain_addendum)} 字符")


[OK] 系统提示词已组装：14897 字符
   core_prompt 长度：6643 字符
   domain_addendum 长度：8247 字符


```text
[OK] 系统提示词已组装：14897 字符
   core_prompt 长度：6643 字符
   domain_addendum 长度：8247 字符
```

&emsp;&emsp;**安装插件与配置 Agent 工厂。** `install_hermes_plugin()` 是幂等操作，把 `submit_query_plan` 和 `lookup_dimension_members` 注册到 Hermes 全局工具注册表。注意这两个工具的参数校验都依赖领域包：`submit_query_plan` 接收的 Intent 由 `IntentValidator` 用 `catalog` 做确定性校验（指标是否登记、维度是否支持）；`lookup_dimension_members` 直接从 `catalog.dimension(id)` 读取取值列表，不是 RAG 检索。`AgentRuntimeConfig` 封装模型调用的连接参数（模型标识、提供商、API 端点、认证密钥、推理强度）。`HermesAgentFactory` 是工厂——它本身不是 Agent，每次 operation 通过 `create_planning_agent()` 创建全新 Agent 实例，并按阶段锁死工具集合（只暴露 planning 工具集）。

In [7]:
# 安装插件 + 配置 Agent Runtime
from hermes_analytics.hermes_adapter.plugin import install_hermes_plugin
from hermes_analytics.hermes_adapter.agent import (
    AgentRuntimeConfig,
    HermesAgentFactory,
)

install_hermes_plugin()  # 幂等操作：把 submit_query_plan / lookup_dimension_members 注册到全局工具注册表
from tools.registry import registry

planning_tools = registry.get_tool_names_for_toolset(
    "hermes_analytics_planning"  # 返回该工具集下注册的工具名列表
)
print(f"[OK] 插件已安装，注册工具：{planning_tools}")

# 凭据已在环境准备块从 .env 加载：api_key / provider / model_name / base_url
config = AgentRuntimeConfig(
    model=model_name,               # 模型标识 ← .env HERMES_ANALYTICS_MODEL_NAME
    provider=provider,              # 提供商 ← .env HERMES_ANALYTICS_MODEL_PROVIDER
    base_url=base_url,              # API 端点 ← .env HERMES_ANALYTICS_MODEL_BASE_URL
    api_key=api_key,                # 认证密钥 ← .env HERMES_ANALYTICS_MODEL_API_KEY（不打印）
    planning_reasoning_effort="low",  # 规划阶段推理强度："none"/"minimal"/"low"/"medium"/"high"/"xhigh"
)
factory = HermesAgentFactory(config)  # 工厂：每次 operation 通过 create_planning_agent() 创建全新 Agent
print(f"[OK] Agent 工厂已就绪：provider={provider}, model={config.model}")

[OK] 插件已安装，注册工具：['lookup_dimension_members', 'submit_query_plan']
[OK] Agent 工厂已就绪：provider=openrouter, model=deepseek/deepseek-v4-flash


```text
# provider/model 随 .env 取值显示；当前 .env 为 openrouter + deepseek/deepseek-v4-flash
[OK] 插件已安装，注册工具：['lookup_dimension_members', 'submit_query_plan']
[OK] Agent 工厂已就绪：provider=openrouter, model=deepseek/deepseek-v4-flash
```

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172951080.png" alt="领域包、系统提示词与 Agent 工厂共同定义能力边界" width="100%"></div>

### 2.2 安全机制：提示词版本化与 Agent 能力锁死

&emsp;&emsp;到这里我们已经有了工厂，但在真正提问之前，还有两个关键问题需要回答：**系统提示词在生产环境中是怎样装配的？** 以及 **Agent 为什么不会越界乱来？** 这两个问题的答案构成了 HermesAnalytics 的安全基线。

&emsp;&emsp;**PromptComposer：三层提示词装配与内容哈希。** 上面我们手动拼接了 `planning_prompt + domain_addendum`，但生产环境中系统提示词由 `PromptComposer` 装配。它把提示词分成三层 XML 标签（`<core_prompt>` + `<domain_addendum>` + `<turn_context>`），每层独立算 SHA-256 哈希，合成整体 `prompt_hash`。这个哈希有双重作用：

&emsp;&emsp;**一是审计追溯与版本绑定。** `prompt_hash` 被写入数据库每条查询记录中（和 `plan_hash`、`sql_hash` 并列）。事后审查某次分析结果时，可以通过它精确还原"这次结果是用哪版提示词、哪个领域包、哪个运行上下文生成的"。改任何一个字段（哪怕只是 `operation_id`），哈希就变，就能区分出不同版本的生成结果。

&emsp;&emsp;**二是帮助比较提示词层是否发生变化。** 三层独立哈希让审计者能判断变化来自 `core_prompt`、`domain_addendum` 还是 `turn_context`；哈希本身不触发、也不保证 Provider 的 Prompt Caching。是否命中缓存，只能依据对应 Provider 在当次响应中返回的缓存指标判断。

&emsp;&emsp;此外，`_reject_raw_user_text()` 会检查 `turn_context` 中是否混入了原始用户文本——如果有就直接报错，防止用户的自然语言绕过结构化约束进入提示词。

In [8]:
# PromptComposer：三层装配 + 内容哈希
from hermes_analytics.hermes_adapter.prompt import (
    PromptComposer,       # 三层提示词装配器
    DomainAddendum,      # 领域包投影数据
    TurnContext,         # 运行上下文（today / data_updated_at 等）
)

composer = PromptComposer(
    core_version="1.0.0",  # 核心提示词版本号
)
bundle = composer.compose(
    phase="planning",      # "planning" 或 "interpretation"
    domain=DomainAddendum(
        version="1.4.0",   # 领域包版本
        domain_name="default-ecommerce",
        content=json.dumps(  # 上面构建的 domain_addendum
            build_planning_domain_content(catalog),
            ensure_ascii=False,
        ),
    ),
    turn_context=TurnContext(
        version="1",
        operation_id="op-001",
        analysis_context={   # 服务端注入的确定性数据
            "today": "2026-08-17",
            "data_updated_at": "2026-06-30",
        },
    ),
)

# bundle 的四个字段见上方文字说明；这里打印哈希做现场确认
print(f"prompt_hash: {bundle.prompt_hash}")
print(f"version: {bundle.version}")
for layer, h in bundle.layer_hashes.items():
    print(f"  {layer}: {h}")

prompt_hash: sha256:05b8e07c8346fe45e17e404a087d2f5c3f8128f4b7ea1202b55fa5b8d7e4ae05
version: 1.0.0+1.4.0+1
  core: sha256:671b455801423aea8d9e22033f44b4edac935747899daae7245529f64717fb9c
  domain: sha256:75ddb4e3fdcf7e7da8889fdf47d90a9b8e292e4fc9b08c21c2fffd9b5ec6b73c
  turn_context: sha256:5734f1bb9779d0f743da567055e119629f7be87b8397a5c1f22d18a5a9ac1c4d


```text
prompt_hash: sha256:05b8e07c8346fe45e17e404a087d2f5c3f8128f4b7ea1202b55fa5b8d7e4ae05
version: 1.0.0+1.4.0+1
  core: sha256:671b455801423aea8d9e22033f44b4edac935747899daae7245529f64717fb9c
  domain: sha256:75ddb4e3fdcf7e7da8889fdf47d90a9b8e292e4fc9b08c21c2fffd9b5ec6b73c
  turn_context: sha256:5734f1bb9779d0f743da567055e119629f7be87b8397a5c1f22d18a5a9ac1c4d
```

&emsp;&emsp;**Agent 安全覆写：为什么 Agent 不会越界。** `HermesAnalyticsAgent` 继承自 Hermes 原生的 `AIAgent`，但通过两组机制把通用 Agent 锁成了只能做数据分析的专用 Agent。

&emsp;&emsp;第一组是 `__init__` 中锁死的 6 个安全参数（外部无法覆盖）：

<br>

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 2-1 Agent 锁死的安全参数</font></p>

<div class="center">

| 参数 | 值 | 作用 |
| :--- | :---: | :--- |
| `skip_context_files` | `True` | 禁止加载 Hermes 默认上下文文件，模型不会读到你的文件系统 |
| `skip_memory` | `True` | 禁止使用 Hermes 记忆系统，模型不会记住上一轮对话偷偷用 |
| `save_trajectories` | `False` | 禁止保存对话轨迹到磁盘，不留敏感数据痕迹 |
| `quiet_mode` | `True` | 静默模式，不打印内部调试日志 |
| `session_db` | `None` | 不使用会话数据库，Agent 无持久化状态 |
| `max_iterations` | `8` | 最多 8 轮工具调用后强制停止，防止无限循环 |

</div>

<br>

&emsp;&emsp;第二组是覆写的 5 个关键方法，控制 Agent 的行为边界：

<br>

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 2-2 Agent 覆写的行为控制方法</font></p>

<div class="center">

| 方法 | 覆写效果 |
| :--- | :--- |
| `_require_system_message()` | 强制要求每次调用都提供系统提示词，不允许空提示词运行 |
| `_build_system_prompt()` | 只用我们提供的提示词，不拼接 Hermes 默认的 "You are Hermes Agent" 身份 |
| `_execute_tool_calls()` | 提交类工具执行成功后立即触发 `PHASE_SUBMISSION_COMPLETE` 停止 Agent，不让模型再开口浪费一次 API 调用 |
| `_handle_max_iterations()` | 迭代预算耗尽时返回阶段完成语，不再向 Provider 请求无意义的摘要 |
| `_emit_status()` | 过滤掉 "asking model to summarise" 等误导信息，避免干扰诊断 |

</div>

&emsp;&emsp;这两组机制合在一起，回答了学员最常问的两个问题："模型会不会读到我的文件系统？"（不会，`skip_context_files=True`）；"模型会不会记住对话偷偷用？"（不会，`skip_memory=True`）。Agent 的唯一对外通道就是 `submit_query_plan` 工具，其他一切能力都被锁死了。

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819173120783.png" alt="PromptComposer 三层提示词与 Agent 安全锁" width="100%"></div>

In [9]:
# Agent 安全覆写：验证表 2-1 / 表 2-2 中锁死的行为
from hermes_analytics.hermes_adapter.agent import (
    PHASE_MAX_ITERATIONS,
    _SUBMISSION_TOOLS,
)

# 参数清单见表 2-1；这里只打印数值做现场确认
print(f"最大迭代次数: {PHASE_MAX_ITERATIONS}")

# 提交类工具清单：命中即触发 PHASE_SUBMISSION_COMPLETE（见表 2-2 第三行）
print(f"提交后停止的工具: {_SUBMISSION_TOOLS}")

# 身份替换验证：验证表 2-2 第二行——提示词原样生效，不拼接默认身份
# 创建 Agent 需要 .env 已配置模型凭据（工厂在无 Key 时直接拒绝创建）
if not api_key:
    print("[SKIP] .env 未配置模型凭据，跳过身份替换验证")
else:
    agent = factory.create_planning_agent(session_id="demo")
    our_prompt = agent._build_system_prompt("HermesAnalytics 数据分析师")
    print(f"提示词等于原始输入: {our_prompt == 'HermesAnalytics 数据分析师'}")
    print(f"包含 Hermes 默认身份: {'You are Hermes Agent' in our_prompt}")

最大迭代次数: 8
提交后停止的工具: {'publish_attribution_advocate_position', 'request_next_attribution_round', 'submit_query_plan', 'publish_attribution_skeptic_position', 'publish_attribution_report', 'publish_analysis_answer'}
提示词等于原始输入: True
包含 Hermes 默认身份: False


&emsp;&emsp;下面代码块展示的是**未配置 `.env` 时的分支示例**。本 Notebook 当前保存的上一格输出来自已配置凭据的历史运行，因此会看到 `提示词等于原始输入: True` 与 `包含 Hermes 默认身份: False`；重新运行时以你本机实际分支为准。

```text
最大迭代次数: 8
提交后停止的工具: {'submit_query_plan', 'publish_analysis_answer', ...}
[SKIP] .env 未配置模型凭据，跳过身份替换验证
```

### 2.3 实际调用：观察模型的原始返回

&emsp;&emsp;前面两个小节准备好了所有基础设施：领域包、提示词、Agent 工厂、安全机制。现在用 `factory.create_planning_agent()` 创建一个真正的 Agent 实例，通过 `run_conversation()` 直接向模型提问——不需要 Runner、Router 或 Core，一次调用就能看到模型返回了什么。

&emsp;&emsp;我们用"哪个渠道表现最好？"这个问题实际调用一次。因为这个问题缺少指标和时间范围，预期模型会发起澄清而不是直接回答：

In [ ]:
# Agent 提问演示：直接调用 run_conversation() 观察模型返回
# factory.create_planning_agent(session_id) 创建 Agent 实例
# agent.run_conversation(user_message, system_message, task_id) 发起一次完整调用
if not api_key:
    print("[SKIP] .env 未配置模型凭据，跳过 Agent 提问演示")
else:
    agent = factory.create_planning_agent(
        session_id="demo-session-001"  # 会话标识，区分不同用户上下文
    )
    result = agent.run_conversation(
        user_message="哪个渠道表现最好？",  # 用户原始问题
        system_message=system_prompt,       # 2.1 中组装的系统提示词
        task_id="demo-op-001",             # operation 标识
    )

    # result 字典：final_response（文本回复）/ messages（含工具调用的历史）/ api_calls（请求数）
    print(f"[OK] Agent 调用完成")
    print(f"   final_response: {result.get('final_response', '')[:200]}")
    print(f"   api_calls: {result.get('api_calls', '?')}")
    print(f"   cache_read_tokens: {result.get('cache_read_tokens', 0)}")

    # 遍历消息历史，找到模型发起的工具调用
    for msg in result.get("messages", []):
        if msg.get("tool_calls"):
            for tc in msg["tool_calls"]:
                fn = tc.get("function", {})
                print(f"   工具调用: {fn.get('name')}")
                print(f"   参数: {fn.get('arguments', '')[:300]}")

```text
[OK] Agent 调用完成
   final_response: 当前系统会话已失效或未正确绑定，导致我无法提交查询计划。...
   api_calls: 5
   cache_read_tokens: 7840
   工具调用: submit_query_plan
   参数: {"kind": "clarification", "clarification": "{\"reason\": \"未指定评价指标\", \"field\": \"metric\", ...}"}
   [工具返回] {"error":{"code":"TOOL_REJECTED","message":"工具调用未通过约束。","reason":"Hermes operation 已失效或未绑定"}}
```

&emsp;&emsp;这段真实输出值得逐行拆解，它同时暴露了三层信息：

&emsp;&emsp;**第一层：模型的行为是对的。** 它没有直接回答"哪个渠道表现最好"，而是调用 `submit_query_plan` 提交了 `kind=clarification` 的结构化 JSON——因为它判断出这个问题缺少指标。这就是课件反复强调的核心机制：**模型不写 SQL、不直接回答，它只提交结构化 Intent，后续由确定性程序处理。**

&emsp;&emsp;**第二层：工具被拒了。** 注意 `[工具返回]` 那行：`TOOL_REJECTED`，原因是 `Hermes operation 已失效或未绑定`。这不是模型的错——2.3 的裸调用没有把任何 Core 绑定到这个 operation，工具 handler 找不到接收方，只能拒绝。模型随后反复重试了 5 次（`api_calls: 5`），每次都被拒。**缺的不是模型能力，是那根"接线"**——这正是第三章要补的东西：`DemoPlanningCore` + `GLOBAL_TOOL_ROUTER.bind()`。

&emsp;&emsp;**第三层：Provider 报告了缓存读取。** `cache_read_tokens: 7840` 仅表示该次 Provider 响应报告了缓存读取；它不由 `prompt_hash` 归因，也不保证其他 Provider、模型或请求都会命中。

&emsp;&emsp;到这里，我们已经跑通了 Agent 调用的全部基础环节：领域包加载、系统提示词组装、插件安装、Agent 工厂创建、提示词版本化、Agent 能力锁死，并且实际看到了模型如何调用工具、提交结构化澄清。

&emsp;&emsp;但 Agent 提交的 Intent 怎样进入确定性主链？完整的衔接代码（DemoPlanningCore + 离线验证 + Agent 实际调用）会在学完领域包地基之后演示——那时你已经理解了 manifest.json 的结构，能看懂 Core 内部的每一步。下面进入第三章，把这条链路的“地基”讲清楚。

## <center>第三章：领域包与 Agent 完整调用</center>

&emsp;&emsp;在主链运行中，脚本 `load_file(manifest.json)` 读到的那份 1347 行 JSON，就是本课所有判断的地基。它回答了 NL2SQL 最容易被问倒的问题：**模型凭什么知道有哪些表、哪些字段、哪些指标，又凭什么选对这一张表和这一个字段？** 这一节先把它讲清，后续的澄清与语义边界才有落点。它是五步主线之前的语义地基，不是五步之一。

&emsp;&emsp;**第一步　先问一个问题：模型是在哪里知道这张表的？**

&emsp;&emsp;回到主链运行：你提的问题要查 `analytics.v_paid_order_items` 这张表，最终生成的 SQL 里也出现了这张表。要引用一张表，总得先知道它有这个名字——那么，模型是在哪里知道它的？想一想，你脑子里冒出的第一个答案是什么。

&emsp;&emsp;最常见的三个答案是：训练数据里见过；RAG 检索出来的 schema；写死在提示词里。**三者都不是**——这个项目的模型从头到尾不知道表名，它只提交一个字符串 `paid_orders`（指标 ID）；至于这个指标落在哪张表、哪个字段，是程序查领域包登记查出来的。先记住这个结论，下面四步把它拆开。

&emsp;&emsp;**第二步　领域包三件套：表、指标、维度都登记在 manifest.json**

&emsp;&emsp;领域包就是 `database/domain/default-ecommerce/manifest.json`，当前登记了三类核心对象，每一类都把「业务语言」和「物理表字段」绑在一起（由 `nl2sql/catalog/loader.py` 的 `DomainPackageLoader` 负责加载校验）：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-5　领域包三件套登记了什么</font></p>

<div class="center">

| 对象 | 数量 | 关键字段 | 绑定关系 |
| :---: | :---: | :---: | :---: |
| 治理视图 views | 6 | `name`（表名）、`grain`（每行粒度）、`fields[].name/type/semantic_type/key_role` | 一张可查询的表 + 每个字段的业务语义 |
| 指标 metrics | 5 | `id`、`display_name`、`description`、`synonyms`、`formula.sources/components/alignment`、`dimensions` | 一个指标 = 业务口径 + 计算公式 + 数据来源 + 支持的维度 |
| 维度 dimensions | 4 | `id`、`levels[].source_view + field + key_field` | 一个维度 = 它落在哪张表的哪个字段 |

</div>

&emsp;&emsp;注意指标公式里的 `sources[].source_view`、`components[].field`，以及维度里的 `levels[].source_view + field`——这些结构性登记才是「选表选字段」的依据，后面会具体用到。除这三件套外，领域包还登记了函数白名单（8 个）、单位（4 个）、查询预算、参数集合，以及 4 个独立版本号。一句话：**模型能问什么、问到什么粒度，边界都画在这份 JSON 里，不在模型脑子里。**

&emsp;&emsp;**第三步　Schema 怎么进模型：三层提示词 + 一个读值工具**

&emsp;&emsp;模型不读数据库、也不读这份 manifest 原文，它读的是被投影进系统提示词的 `<domain_addendum>`。装配链是：`build_planning_domain_content`（`application/analysis_planning/domain_content.py`）先把领域包投影成精简 JSON；`PromptComposer.compose`（`hermes_adapter/prompt.py`）再把它和核心提示词、运行上下文拼成三层——`<core_prompt>`（能力槽位）+ `<domain_addendum>`（领域包投影）+ `<turn_context>`（服务端确定性数据），三层各自算一个哈希，合成整体 `prompt_hash`，只用于版本绑定与审计，不会隐式触发 Prompt Caching。核心提示词里的 `<<comparisons>>`、`<<ranking_fields>>` 这类槽位由 `nl2sql/capabilities.py` 的能力清单在装配期填入——**填不满就当场报错**，绝不让一个空槽位漏进提示词。

&emsp;&emsp;维度取值一样走「登记」：取值不超过 `member_inline_limit`（当前 50）时直接内联进提示词；超过时只发取值条数，模型必须调用 `lookup_dimension_members` 工具把取值读全再填筛选。这个工具的源码注释写得很死——「这不是检索：没有关键词、模糊匹配、分页、排序，入参只有两个已登记 ID」。它是按 ID 读值，不是语义检索，这是本项目和 RAG 最直接的一刀切。

&emsp;&emsp;**先把一个容易误解的字段讲透：`description` 不是选表映射**

&emsp;&emsp;指标 `description` 既不是无用注释，也不是隐藏的 SQL 公式。`DomainPackageLoader` 会读取并校验整份 JSON，形成运行时 Catalog；规划阶段再从 Catalog 投影 Hermes 需要的内容，其中明确包含每个指标的 `description`。同一份说明在后续有三类去向，责任并不相同：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 0-6　`description` 的三类去向与边界</font></p>

<div class="center">

| 去向 | 实际用途 | 明确不负责什么 |
| :---: | :---: | :---: |
| 规划阶段 Hermes | 随指标 ID、展示名、别名等进入 `<domain_addendum>`，帮助 Hermes 理解业务口径并提交已登记的指标 ID | 不决定物理表、字段、Join 或 SQL |
| `ClarificationDetector` 候选逻辑 | 在 `_fuzzy_match_tier` 中作为 `tier=2` 的模糊候选证据；展示名是 tier 0，别名是 tier 1 | 不把模糊文本自动改写成指标 ID，也不替代后续确定性校验 |
| 冻结计划与结果口径 | `QueryPlanner` 把它写入 `MetricPresentation.definition`，再进入 `QueryPlan.metric_definition`，供冻结计划、结果解释与口径展示 | 不替代公式、字段契约、Policy 或结果正确性验证 |

</div>

&emsp;&emsp;可以把 `description` 理解成商品旁边的「说明牌」：它告诉模型、程序和人这项指标在业务上是什么意思，也能在表达含糊时提供候选线索；但说明牌不是仓库货位图，不能指挥系统去哪个表、拿哪个字段。

&emsp;&emsp;例如，`net_sales` 的描述写着「实付商品金额减去退款商品金额」，不能因此直接推断 SQL 必须读取 `paid_amount` 和 `refunded_amount`。真正的物理映射要看 manifest 原始结构中的 `formula.sources`、`formula.components`、维度 `levels` 和治理视图字段契约；多源指标还受 `alignment` 与已登记 Join 的约束。运行时的 `formula.source_views` 是从这些来源声明派生出的属性，不是 manifest 中手写的原始字段。

&emsp;&emsp;**源码锚点**：整包加载与校验见 [`loader.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/catalog/loader.py)，规划投影见 [`domain_content.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_planning/domain_content.py)，Prompt 装配见 [`prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py)，候选分级见 [`service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/intent/service.py)，冻结口径与物理规划见 [`planner.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/planning/planner.py)。

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172951619.png" alt="模型只提交指标与维度 ID，编译器查领域包登记确定性选表选字段" width="100%"></div>

&emsp;&emsp;**第四步　谁决定用哪张表哪个字段：模型不选，编译器选**

&emsp;&emsp;这是最关键的一步。模型提交的 Intent 里只有 `metric="net_sales"`、`group_by=("channel",)` 这样的 ID，没有任何表名。真正选表的是 `QueryPlanner.plan`（`nl2sql/planning/planner.py`）里的一行：

```text
source_views = metric.formula.source_views   # 用哪张表，由指标的公式登记决定
```

&emsp;&emsp;即：净销售额的 `formula.sources` 在 manifest 中分别登记了支付与退款来源视图，运行时再由它们派生出 `formula.source_views`；渠道维度则由 `levels[].source_view + field` 登记落在哪张表的哪个字段。**模型只负责「问题 → 指标 ID + 维度 ID」，表与字段的映射是编译器查领域包登记、确定性拿到的。** 这就是没有训练数据、没有 RAG 也能选对表的根因——不需要「学」映射，因为映射早已显式登记，编译器只是照章办事。

&emsp;&emsp;**第五步　为什么不训练、不 RAG，数据表多了怎么办**

&emsp;&emsp;弄清楚「模型不选表」之后，再回头看传统 NL2SQL 的两条路就看得清了。传统做法要让模型写出完整 SQL，业务知识靠两条路补：路 A 是用成对的「(问题, SQL)」训练数据微调，让模型自己把「净销售额」和某张表某几列对上；路 B 是把 schema 向量化，用 RAG 检索出「最像的几张表」喂给模型。两条路各有代价——训练要标注、换域重训，RAG 会召回错表、要维护向量库。

&emsp;&emsp;本项目因为表字段映射早已显式登记、是确定性查表，所以**既不需要训练数据、也不需要检索**。但第三条路只在封闭企业域里划算：能问的指标和维度有限且确定，登记得过来；换成开放领域自由问答，登记制撑不住，那时训练或 RAG 才必要——上面第三步的 `lookup_dimension_members`（读值不是检索）就是这条边界的证据。

&emsp;&emsp;那么数据表多了怎么办？不需要改代码，只需要在领域包里多登记对应的 `views`/`metrics`/`dimensions`。领域包本身就是版本化的（`default-ecommerce@1.4.0` + 四个独立版本号），`content_hash` 参与 `plan_hash`——当前这份领域包的内容哈希是 `9f8def6d...`（`catalog.content_hash` 可现场打印），领域包一变哈希跟着变。规模大到不适合「整包全量注入」时，`build_planning_domain_content` 的注释已留好扩展点：「…若要按问题裁剪指标与维度，改这里即可，主链和 Prompt 装配不需要改动」；编译器「会不会写某种 SQL」的硬边界由 `nl2sql/capabilities.py` 能力清单守住、启动自检拦截不一致。至于「谁有权、怎样安全地改这份 manifest」，是后续课程要回答的问题——**模型能问什么，来自这份登记；而这份登记，专人负责变更。**

&emsp;&emsp;**本板块自测**：主链跑出的 SQL 里出现了 `analytics.v_paid_order_items` 这张表，它是模型猜出来的，还是编译器查出来的？答案：编译器查出来的——`paid_orders` 指标在 manifest 的 `formula.sources[].source_view` 中登记了这张表，运行时再派生出 `formula.source_views`；模型从未「选择」过表名，它只是说「我要支付订单数、按渠道分」。

&emsp;&emsp;下一节问题：业务语义已经登记在领域包里，但当模型交出的 Intent 缺字段、或组合超出登记范围时，系统会反问还是拒绝？

**Agent 完整调用演示：从提交到确定性链**

&emsp;&emsp;前面我们已经分别跑通了确定性主链（`mainline.py`）和 Agent 基础能力（工具注册、提示词、工厂）。现在把两段接起来：用 `DemoPlanningCore` 作为"接头"，让 Agent 提交的 Intent 直接进入确定性主链。这一节是 2.3 结尾留下的悬念的答案——被拒 5 次的工具调用，缺的就是这里的三样东西：一个接收提交的 Core、一次路由绑定、一个受超时保护的运行器。

&emsp;&emsp;以下三个小节接续前面的变量（`catalog`、`system_prompt`、`factory`、`api_key`），按顺序执行。

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172952672.png" alt="运行时接线：路由绑定把 Agent 工具调用接入 DemoPlanningCore，查询与澄清双分支严格分流" width="100%"></div>

### 3.1 实现接收提交的 Core

**第一步：实现 DemoPlanningCore。** 这是整个链条的"接头"——Agent 调用 `submit_query_plan` 工具时，工具 handler 会把结果写入这个 Core 槽位。它接收 `PlanSubmission`，然后走完整条确定性主链（Intent -> SQL -> Policy -> FrozenQuery）。

In [10]:
# 3.1：实现 DemoPlanningCore —— Agent 提交的"接头"
from datetime import date
from typing import Any
from hermes_analytics.hermes_adapter.port import PlanSubmission, MemberLookup
from hermes_analytics.nl2sql.intent.models import (
    AnalysisIntent,
    TimeRange as IntentTimeRange,
)
from hermes_analytics.nl2sql.intent.service import (
    IntentValidator,
    IntentValidationError,
)
from hermes_analytics.nl2sql.planning.planner import QueryPlanner
from hermes_analytics.nl2sql.compilation.compiler import SqlCompiler
from hermes_analytics.nl2sql.policy.inspector import SqlPolicy
from hermes_analytics.nl2sql.compilation.freeze import FrozenQueryFactory


class DemoPlanningCore:
    """演示用 PlanningCorePort 实现：接收 PlanSubmission，走完整条确定性主链。

    澄清提交（kind=clarification）只登记不进链；查询提交（kind=query）
    逐条校验并编译 Intent -> SQL -> Policy -> plan_hash。
    """

    def __init__(self) -> None:
        """初始化演示过程中的提交、SQL 与哈希记录。

        Returns:
            None。
        """
        self.submissions: list[PlanSubmission] = []
        self.compiled_sql: list[str] = []
        self.plan_hashes: list[str] = []

    def submit_query_plan(self, submission: PlanSubmission) -> dict[str, Any]:
        """处理一次查询计划提交。

        Args:
            submission: Agent 提交的查询或澄清结构。

        Returns:
            包含发布状态的字典。
        """
        self.submissions.append(submission)
        print(f"\n[Core] 收到提交：kind={submission.kind}")

        if submission.kind == "clarification":
            c = submission.clarification or {}  # 澄清请求：reason / field / question / candidate_ids
            print(f"   [Clarification] 问题：{c.get('question', '?')}")
            print(
                f"   [Clarification] 原因：{c.get('reason')}，"
                f"字段：{c.get('field')}"
            )
            return {"status": "published"}  # 澄清不进入确定性主链

        if submission.kind == "query":
            validator = IntentValidator()  # 无参构造；validate(intent, catalog) 做确定性校验
            for i, intent_dict in enumerate(submission.query_intents):
                print(f"\n   -- Intent {i + 1} --")
                print(f"   指标：{intent_dict.get('metric')}")
                tr = intent_dict.get("time_range") or {}
                print(
                    f"   时间：{tr.get('start', '?')} ~ "
                    f"{tr.get('end', '?')}"
                )
                print(f"   分组：{intent_dict.get('group_by')}")

                try:
                    intent = AnalysisIntent(  # 七字段结构化意图：metric / time_range / time_grain / group_by / filters / comparison / ranking
                        metric=intent_dict.get("metric"),
                        time_range=IntentTimeRange(
                            start=date.fromisoformat(tr["start"]),
                            end=date.fromisoformat(tr["end"]),
                        ) if tr.get("start") and tr.get("end")
                        else None,
                        time_grain=intent_dict.get("time_grain"),
                        group_by=tuple(intent_dict.get("group_by") or ()),
                        filters=tuple(intent_dict.get("filters") or ()),
                        comparison=intent_dict.get("comparison", "none"),
                        ranking=intent_dict.get("ranking"),  # dict 直传，Pydantic 自动转 Ranking；Top N 保留进 SQL 的 LIMIT
                    )
                except Exception:
                    print("   [ERROR] Intent 构造失败")
                    continue

                try:
                    validated = validator.validate(intent, catalog)  # 校验指标登记、时间合法性、维度支持、筛选算子；失败抛 IntentValidationError
                    print(f"   [VALID] metric={validated.metric}")
                except IntentValidationError as exc:
                    print(f"   [INVALID] {exc.code}（{exc.field}）")
                    continue

                plan = QueryPlanner().plan(validated, catalog)       # Intent -> QueryPlan（可重放计划）
                compiled = SqlCompiler().compile(plan, catalog, catalog.content_hash)  # QueryPlan -> 参数化 SQL
                policy = SqlPolicy().inspect(compiled.sql, plan, catalog)  # SQLGlot 白名单安全检查
                frozen = FrozenQueryFactory().freeze(compiled, plan, catalog)  # plan_hash 冻结（内容寻址防篡改）

                self.compiled_sql.append(compiled.sql)
                self.plan_hashes.append(frozen.plan_hash)

                print(f"   [SQL] {compiled.sql.strip()[:80]}...")
                print(
                    f"   [Policy] allowed={policy.allowed}"
                    f" ({[d.rule_id for d in policy.decisions]})"
                )
                print(f"   [Hash] {frozen.plan_hash}")

            return {"status": "published"}

        return {"status": "published"}

    def lookup_dimension_members(self, lookup: MemberLookup) -> dict[str, Any]:
        """按已登记 ID 读取维度取值。

        Args:
            lookup: 维度与层级的精确 ID。

        Returns:
            取值列表或机器可读错误字典。
        """
        dim = next(
            (d for d in catalog.manifest.dimensions
             if d.id == lookup.dimension_id),
            None,
        )
        if dim is None:  # 维度 ID 不在登记里：回已知 ID 列表，模型可纠正
            return {"error": {
                "code": "UNKNOWN_DIMENSION_ID",
                "known_dimension_ids": [
                    d.id for d in catalog.manifest.dimensions
                ],
            }}
        level = next(
            (lv for lv in dim.levels if lv.id == lookup.level_id),
            None,
        )
        if level is None:  # 层级 ID 不在该维度下：回该维度的已知层级
            return {"error": {
                "code": "UNKNOWN_LEVEL_ID",
                "dimension_id": dim.id,
                "known_level_ids": [lv.id for lv in dim.levels],
            }}
        return {
            "status": "MEMBERS_LISTED",
            "members": [
                {"key": m.key, "label": m.label}  # key 写入 filters，label 给人看
                for m in (level.members or ())
            ],
        }


core = DemoPlanningCore()
print("[OK] DemoPlanningCore 已创建")


[OK] DemoPlanningCore 已创建


In [11]:
# 3.1 续：现场验证 lookup 的三种结局（生产 bridge.py 同款返回结构）
good = core.lookup_dimension_members(
    MemberLookup(operation_id="t1", dimension_id="channel", level_id="channel")
)
print(f"正确 ID  -> 状态: {good.get('status')}，取值数: {len(good.get('members', []))}")

bad_dim = core.lookup_dimension_members(
    MemberLookup(operation_id="t2", dimension_id="chanel", level_id="channel")  # 拼错的维度 ID
)
print(f"错维度   -> {bad_dim.get('error', {}).get('code')}，已知维度: {bad_dim.get('error', {}).get('known_dimension_ids')}")


正确 ID  -> 状态: MEMBERS_LISTED，取值数: 6
错维度   -> UNKNOWN_DIMENSION_ID，已知维度: ['time', 'channel', 'product', 'region']


```text
# MEMBERS_LISTED=取值已列出 UNKNOWN_DIMENSION_ID=维度 ID 不在登记里（回已知列表供模型纠正）
[OK] DemoPlanningCore 已创建
正确 ID  -> 状态: MEMBERS_LISTED，取值数: 6
错维度   -> UNKNOWN_DIMENSION_ID，已知维度: ['time', 'channel', 'product', 'region']
```

### 3.2 确定性链离线验证

**第二步：确定性链离线验证。** 在真正调用 Agent 之前，先模拟一个完整 Intent 的提交，确认 Core 的确定性主链能正常工作——不依赖模型、不依赖网络。这一步的价值在于把"模型出错"和"主链出错"两类故障分开：如果这一步都跑不通，问题一定在确定性代码里，与模型无关。

In [12]:
# 3.2：确定性链离线验证（不调 Agent，不依赖网络）
result = core.submit_query_plan(
    PlanSubmission(
        operation_id="test-offline",  # 服务端 operation 标识
        kind="query",                 # "query" | "clarification" | "history_selection"
        query_intents=(               # 查询意图字典元组
            {
                "metric": "paid_orders",
                "time_range": {
                    "start": "2026-06-01",
                    "end": "2026-06-30",
                },
                "time_grain": None,
                "group_by": ["channel"],
                "filters": [],
                "comparison": "none",
                "ranking": None,
            },
        ),
    )
)
assert result == {"status": "published"}
assert len(core.compiled_sql) == 1
print(f"\n[OK] 确定性链离线验证通过")
print(f"   SQL: {core.compiled_sql[0].strip()[:100]}...")
print(f"   plan_hash: {core.plan_hashes[0]}")



[Core] 收到提交：kind=query

   -- Intent 1 --
   指标：paid_orders
   时间：2026-06-01 ~ 2026-06-30
   分组：['channel']
   [VALID] metric=paid_orders
   [SQL] SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_...
   [Policy] allowed=True (['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS'])
   [Hash] sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83

[OK] 确定性链离线验证通过
   SQL: SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_ord...
   plan_hash: sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83


```text
[Core] 收到提交：kind=query

   -- Intent 1 --
   指标：paid_orders
   时间：2026-06-01 ~ 2026-06-30
   分组：['channel']
   [VALID] metric=paid_orders
   [SQL] SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_...
   [Policy] allowed=True (['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS'])
   [Hash] sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83

[OK] 确定性链离线验证通过
   SQL: SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_ord...
   plan_hash: sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83
```

### 3.3 路由绑定与真实调用

&emsp;&emsp;**第三步：Agent 调用。** 这是真正的模型调用（按 2.1 配置的 provider，如 DeepSeek 或 OpenRouter）。本块会连问两个问题做对照：完整问题"2026年6月各渠道支付订单数前十名"预期走查询分支，模糊问题"哪个渠道表现最好？"预期走澄清分支。如果 .env 未配置模型凭据，本块会跳过。

&emsp;&emsp;本块出现的三样新东西，正好对应 2.3 失败运行的三个缺口：

&emsp;&emsp;**`HermesToolPorts(planning=core)`** 把 Core 包成工具端口。2.3 里工具 handler 找不到接收方，因为 `planning` 端口是空的；现在它指向 `core`。

&emsp;&emsp;**`GLOBAL_TOOL_ROUTER.bind(operation_id, ports)`** 把端口绑定到本次 operation。工具 handler 收到调用时，拿 `task_id`（即 `operation_id`）去路由表 resolve 出端口——这就是 2.3 报错 `Hermes operation 已失效或未绑定` 里"绑定"二字的实体。`bind` 是上下文管理器，离开 `with` 块自动解绑，下次 operation 不会拿到上一次的 Core。

&emsp;&emsp;**`BoundedHermesRunner`** 在线程池里承载同步的 Agent 调用并施加超时。直播演示时如果模型卡住，是它（而非演示者）负责在 `timeout_seconds` 后协作式中断——这正是第六章"受控复核"里执行超时控制的同款思想，先在这里见一次。

In [ ]:
# 3.3：Agent 调用（凭据来自 .env，未配置则跳过）
import asyncio
from hermes_analytics.hermes_adapter.port import (
    GLOBAL_TOOL_ROUTER,
    HermesToolPorts,
)
from hermes_analytics.hermes_adapter.runner import BoundedHermesRunner

ports = HermesToolPorts(
    planning=core,  # PlanningCorePort：submit_query_plan / lookup_dimension_members
)
runner = BoundedHermesRunner(
    max_workers=1,              # 线程池大小
    interrupt_grace_seconds=2.0,  # 超时后等待 Agent 协作式中断的宽限期（秒）
)
questions = [
    "2026年6月各渠道支付订单数前十名",  # 完整问题：指标+时间+维度+Top N 齐备，预期走查询分支
    "哪个渠道表现最好？",               # 模糊问题：缺指标和时间，预期走澄清分支
]

if not api_key:
    print("[SKIP] .env 未配置模型凭据，跳过 Agent 调用")
    print("   请检查 HermesAnalytics第二版/.env 的四个模型变量后重新运行本块")
else:
    try:
        for i, user_question in enumerate(questions):
            operation_id = f"demo-operation-{i + 1:03d}"  # 每个问题独立 operation，绑定互不干扰
            before_sql = len(core.compiled_sql)           # 记录循环前数量，统计"本次"生成数
            before_sub = len(core.submissions)

            print(f"\n{'=' * 60}")
            print(f"[Agent] 第 {i + 1} 次调用")
            print(f"   用户问题：{user_question}")
            print(f"{'=' * 60}")

            with GLOBAL_TOOL_ROUTER.bind(operation_id, ports):  # 绑定到本次 operation；离开 with 自动解绑
                agent = factory.create_planning_agent(
                    session_id=f"demo-session-{i + 1:03d}"  # 每次调用新建 Agent，无跨轮状态
                )

                async def _run() -> dict[str, Any]:
                    """在超时保护下执行当前 Agent 请求。

                    Returns:
                        Agent 运行结果字典。
                    """
                    return await runner.run(
                        agent,
                        timeout_seconds=120,       # 单次调用超时上限（秒）
                        user_message=user_question, # 用户原始问题
                        system_message=system_prompt,  # 2.1 组装的系统提示词
                        task_id=operation_id,      # 与 bind 时的 operation_id 一致
                    )

                result = asyncio.run(_run())

            print(f"\n[OK] Agent 调用完成")
            print(
                f"   final_response: "
                f"{result.get('final_response', '')[:120]}"
            )
            print(f"   api_calls: {result.get('api_calls', '?')}")
            print(f"\n[Summary]")
            print(f"   本次提交数：{len(core.submissions) - before_sub}")
            print(f"   本次生成 SQL 数：{len(core.compiled_sql) - before_sql}")
            for j in range(before_sql, len(core.compiled_sql)):
                print(f"   [{j + 1}] {core.plan_hashes[j]}")
                print(f"       SQL: {core.compiled_sql[j].strip()[:100]}...")
    except Exception as exc:
        print(f"\n[ERROR] Agent 调用失败：{exc}")
        print("   请检查：")
        print("   1. .env 中 HERMES_ANALYTICS_MODEL_API_KEY 是否有效（余额充足、未过期）")
        print("   2. 网络是否能访问 .env 所配置 provider 的 API 端点")
    finally:
        runner.close()

print("\n[OK] 3.3 完成")


&emsp;&emsp;下面是成功路径的**示例输出**，用于说明两个分支的预期形状；它不是本次修改后的重新执行结果。真实调用以你当前的模型凭据、网络与 Provider 响应为准。

```text
============================================================
[Agent] 第 1 次调用
   用户问题：2026年6月各渠道支付订单数前十名
============================================================

[Core] 收到提交：kind=query

   -- Intent 1 --
   [VALID] metric=paid_orders
   [SQL] SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_...
   [Policy] allowed=True (['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS'])
   [Hash] sha256:e80d03d0d4032772ea5a1503878bc8ee2ec78e52ce348c9c7107322d5ebb9944
Tool guardrail halted submit_query_plan: PHASE_SUBMISSION_COMPLETE

[OK] Agent 调用完成
   final_response: 结构化查询计划已提交，等待 Core 继续处理。
   api_calls: 1

[Summary]
   本次提交数：1
   本次生成 SQL 数：1
   [1] sha256:e80d03d0d4032772ea5a1503878bc8ee2ec78e52ce348c9c7107322d5ebb9944
       SQL: SELECT channel_id, channel_name AS channel_name, COALESCE((COUNT(DISTINCT order_id)), 0) AS paid_ord...

============================================================
[Agent] 第 2 次调用
   用户问题：哪个渠道表现最好？
============================================================

[Core] 收到提交：kind=clarification
   [Clarification] 问题：您希望用哪个指标来衡量渠道表现？
Tool guardrail halted submit_query_plan: PHASE_SUBMISSION_COMPLETE

[OK] Agent 调用完成
   final_response: 结构化查询计划已提交，等待 Core 继续处理。
   api_calls: 1

[Summary]
   本次提交数：1
   本次生成 SQL 数：0

[OK] 3.3 完成
```

&emsp;&emsp;对照 2.3 的失败运行看两次调用：绑定 `GLOBAL_TOOL_ROUTER` 之后，同一个模型，两次都是 `api_calls: 1` 一次完成，只是走了不同的分支——这行 `bind()` 就是 2.3 里被拒 5 次的解药。

&emsp;&emsp;**第 1 次（完整问题）走查询分支**：指标、时间、维度、Top N 四要素齐备，模型直接提交 `kind=query`；Core 校验通过、编译出参数化 SQL、通过白名单、冻结出 `plan_hash`，`PHASE_SUBMISSION_COMPLETE` 立即停止 Agent。注意 `plan_hash` 与第二章 `mainline.py` 跑出的不同（`e80d03...` 对 `d9e334...`）——这次 Intent 带了 Top N，SQL 多了 `ORDER BY ... LIMIT :top_n`，参数多了 `top_n=10`。计划变，哈希就变，这是"内容寻址"的直接验证。

&emsp;&emsp;**第 2 次（模糊问题）走澄清分支**："哪个渠道表现最好"没有说明按什么指标衡量、也没有时间范围，模型不猜口径，提交 `kind=clarification`，问"您希望用哪个指标来衡量渠道表现？"。Core 只登记澄清、不进确定性主链，所以本次 SQL 数为 0。缺字段就问、不替用户猜——这正是第四章"澄清与语义边界"要展开的机制，这里先见一次真人版。

## <center>第四章：澄清与语义边界</center>

&emsp;&emsp;机制地图的第一步是“补全意图”。3.3 第 2 次调用已经出现过这一幕：模型对“哪个渠道表现最好”不猜口径，反问“您希望用哪个指标来衡量渠道表现？”。这一章把那一幕拆开：反问的内容从哪来、什么情况该拒绝而不是反问、什么时候是排障。当模型提交的 Intent 缺少必要字段时，系统应先暴露缺口，而不是猜测后执行。本章回答：系统反问、拒绝、排障分别说明什么？

&emsp;&emsp;先拆掉最容易形成的四个误解：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-1　常见误解与真实机制</font></p>

<div class="center">

| 常见误解 | 真实机制 | 判断边界 |
| :---: | :---: | :---: |
| 模型反问说明它不会问数 | Core 发现 Intent 缺字段，模型把 `gaps` 翻译成澄清问句 | 缺字段才澄清，不替你猜口径 |
| 程序直接生成中文提问 | Core 顶层返回 `INTENT_INCOMPLETE`（意图不完整），`gaps`（缺口清单）列出具体缺口码 | 程序不回读原话，也不生成自然语言 |
| 退款金额率按品类失败说明数据库坏了 | 当前指标已支持的结果维度不含 product/品类 | 结果维度不支持，不等于基础设施故障 |
| 模型避开不支持组合后程序可以放松校验 | 模型输出仍要经过确定性校验 | 模型判断不能替代领域规则 |

</div>

&emsp;&emsp;**源码锚点**：缺口校验看 [`intent/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/intent/service.py)，模型组织澄清看 [`prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py)，业务支持维度看 [`manifest.json`](HermesAnalytics第二版/database/domain/default-ecommerce/manifest.json)，规划边界看 [`bridge.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_planning/bridge.py)。

&emsp;&emsp;**口语类比与反例**：这像填写经营分析申请单——缺字段要补、菜单不支持要换选项、系统打不开才报修。反例是看到“表现最好”便擅自选择净销售额，或把品类维度不支持归为数据库故障。

### 4.1 缺口澄清：模型翻译程序原因码

&emsp;&emsp;先看一段真实登记。`manifest.json` 里每个指标都长这样（摘自 `paid_orders`，有删节）：

```json
{
  "id": "paid_orders",
  "display_name": "支付订单数",
  "description": "分析期内已支付订单 ID 的去重数。",
  "synonyms": ["支付订单数", "订单数"],
  "dimensions": ["time", "channel", "product", "region"],
  "formula": { "...": "..." }
}
```

&emsp;&emsp;注意 `"dimensions"` 这一行——它登记的是**这个指标允许按哪些维度看**。模型提交 Intent 时写的 `group_by=("channel",)`，就是拿去和这行清单对照的。这一章讲的"缺口"和"边界"，判据全部来自这类登记行。

&emsp;&emsp;HermesAnalytics 的架构铁律是：程序不从自然语言做推断。当模型提交的结构化 Intent 缺少必要字段时，Core 内部的 `ClarificationDetector` 先归一化再逐字段检测，给出三态判定。这三种都是**校验状态码**，不是执行结果：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-2　三态校验状态码</font></p>

<div class="center">

| 状态码 | 中文含义 | 什么时候出现 |
| :--- | :--- | :--- |
| `VALID` | 校验通过 | 七字段齐备且全部合法，进入确定性规划 |
| `INVALID` | 校验失败 | 字段齐备但值非法（如分组维度不在该指标的结果维度清单），退回模型但只给笼统提示，精确原因码只进日志 |
| `CLARIFICATION_REQUIRED` | 需要澄清 | 字段缺失（如没写时间范围），退回模型补齐 |

</div>

&emsp;&emsp;有一个容易判错的路由：指标名写错不会走到 `INVALID`——缺口检测先行一步，名称不是精确命中（未登记、命中多个候选、只命中名称片段）一律报 `METRIC_AMBIGUOUS`（指标候选不唯一或未登记），附候选指标清单退回，让模型自纠。`INVALID` 留给另一种情况：字段齐备、取值非法。两种态都会退回模型，差别在信息精度——缺口态给精确缺口码和候选值，`INVALID` 只给一句笼统文案，不向模型泄露内部实现，精确原因码只写入日志。

&emsp;&emsp;出现缺口时，Core 把 `retry_feedback`（重试反馈）退回给模型，里面带两个东西：`INTENT_INCOMPLETE`（意图不完整，总状态码）和 `gaps`（缺口清单）。`gaps` 里的每一项也是一个机器可读的缺口码，常见的三个：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-3　常见缺口码</font></p>

<div class="center">

| 缺口码 | 中文含义 | 缺的是什么 |
| :--- | :--- | :--- |
| `METRIC_REQUIRED` | 缺指标 | 用户没说要看哪个指标（净销售额？订单数？） |
| `TIME_RANGE_REQUIRED` | 缺时间范围 | 用户没说看哪个时间段 |
| `RANKING_LIMIT_REQUIRED` | 缺排名数量 | 用户说“前几名”但没说具体几个 |

</div>

&emsp;&emsp;注意方向：**澄清只能由模型发起**——Core 只把“缺什么”用机器码说清楚退回去，问不问用户、怎么问由模型决定。模型接住缺口码，把它们翻译成用户能理解的中文澄清问题。

&emsp;&emsp;主任务用“哪个渠道表现最好”来观察缺口澄清。条件是：当模型提交的 Intent 缺少指标或时间时，Core 返回相应 `gaps`，模型再发起澄清。这不是固定两轮承诺，也不是系统不会回答，而是当前分析意图缺少必要字段。

&emsp;&emsp;先看责任链：模型提交 Intent → Core 返回 `INTENT_INCOMPLETE` 与 `gaps` → 模型按缺口组织澄清。下面这张机制图标出三层责任，不是运行截图。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172938154.png" alt="澄清责任链机制示意图：模型提交 Intent、Core 返回缺口、模型组织澄清" width="100%"></div>

&emsp;&emsp;下面这张截图标注为“运行截图｜Compose 项目 hermes-analytics-final”，是本次真实澄清对话。它只证明“出现了澄清交互”；具体问句措辞和澄清轮数以运行结果为准，不承诺固定问法或固定轮数。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172943183.png" alt="运行截图：真实澄清对话" width="85%"></div>

&emsp;&emsp;Top N 是一个不抢主线的边界例：“列出表现最好的渠道”即使已有指标和时间，仍可能缺少排名数量；当 `gaps` 出现 Top N 数量缺口时，应先补“前 3 个”或“前 5 个”，不能由系统擅自决定。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-4　澄清责任链的三层分工</font></p>

<div class="center">

| 层 | 动作 | 产出 | 源码位置 |
| :---: | :---: | :---: | :---: |
| Core 检测缺口 | 归一化后逐字段判定，三态分类 | `INTENT_INCOMPLETE` + 机器可读 `gaps` | `ClarificationDetector.validate_or_find_gaps_detailed`，`bridge.py` 约 279 行调用 |
| 模型组织问句 | 接住 `gaps`，翻译成中文澄清 | 用户能理解的自然语言提问 | 规划提示词约束，`prompt.py` |
| 前端澄清卡展示 | 显示模型问句与候选选项 | 学员可读的澄清界面 | `ConversationTurnCard.tsx` |

</div>

&emsp;&emsp;完整 Intent 通过校验后进入的规划、编译、策略与冻结（`bridge.py` 的 `plan_queries`、`compile_queries`、`inspect_policy`），就是第三章 `DemoPlanningCore` 里已经跑通的那条链，这里不再重复。

&emsp;&emsp;这套责任链的关键判断是：程序只返回机器可读缺口，不生成中文问句；模型负责把原因翻译成用户语言。页面上的澄清问句措辞会变化，但“缺什么字段、由谁组织澄清、完整后才进入确定性规划”不会变。

### 4.2 语义边界：指标 × 维度

&emsp;&emsp;分析意图完整后，还需要判断“这个组合是否在领域中成立”。最容易混淆的是把业务语义不支持当成系统故障。用一个真实的用户问题把这件事走一遍。

&emsp;&emsp;**第一步　看现象：这个请求被拒绝了。** 用户问“2026 年 6 月各品类的退款金额率”，系统拒绝；而几乎同样的问题“各渠道的退款金额率”却能正常通过。同一个指标、同一个时间段，只有分组维度不同，结局一个拒一个过——这不像故障，倒像一条规则。真实运行结果：

&emsp;&emsp;同一个指标只换分组维度，结局会相反。先用下面的判定卡固定预期，再运行后续验证代码。

```text
退款金额率 × 品类分组 → METRIC_DIMENSION_UNSUPPORTED（group_by）
退款金额率 × 渠道分组 → VALID
```

&emsp;&emsp;**第二步　查登记：拒绝的依据是 manifest 里的一行。** 打开 `manifest.json`，对比两个指标的 `dimensions` 登记：

```json
// refund_amount_rate（退款金额率）
"dimensions": ["time", "channel", "region"],

// net_sales（净销售额）
"dimensions": ["time", "channel", "product", "region"],
```

&emsp;&emsp;品类（product）在净销售额的清单里，不在退款金额率的清单里。校验器做的事情就是查这两行清单——`group_by=("product",)` 拿去对照 `refund_amount_rate` 的清单，查不到，返回 `METRIC_DIMENSION_UNSUPPORTED`（指标不支持该维度，机器判定的拒绝码）。判定不经过模型理解，只经过这行登记。

&emsp;&emsp;业务上为什么这么登记？先排除两个错误答案：不是数据没有品类（退款表和支付表都登记了 `category_id` 等字段），也不是 SQL 写不出（绕过校验器直接调规划器和编译器，这条 SQL 能正常生成——拦截只发生在校验这一层）。真实的原因是**口径治理的取舍**：

&emsp;&emsp;退款金额率的分母锚定支付视图（`alignment: anchor_left`），按品类分组时，“有退款、没支付”的品类不会出现在锚点左侧，比率的分母静默缺失，算出来的数字看着正常、实际失真。对照净销售额：它是差值指标，分组丢行时补 0 语义无损（登记 `empty_value: zero`），所以敢开放品类；退款金额率是比率指标，登记的是 `empty_value: null`——**开不开放某个维度，跟着指标的计算语义走，这是登记时的主动决定，不是系统能力做不到。**

<div align="center"><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172953266.png" alt="指标计算语义决定维度白名单：比率指标排除品类防分母失真，差值指标补零无损开放品类" width="100%"></div>

&emsp;&emsp;还有一个学员常问的追问：“那先筛渠道、再按品类算行不行？”——也不行，而且是两道闸：分组闸拒绝品类（`METRIC_DIMENSION_UNSUPPORTED`），筛选闸连渠道维度 ID 都不认（筛选的合法字段从该指标 `dimensions` 展开的字段名里取，写 `channel` 不在、写 `channel_name` 才在）。**dimensions 清单同时管分组和筛选**，这就是“指标视角白名单”的含义。

&emsp;&emsp;**第三步　定位：这属于三分法里的哪一类。** 系统、数据库、SQL 引擎此刻都正常，校验器拒绝这个组合恰恰是它的**正确行为**。对照三分法：这不是意图缺口（字段不缺），不是基础设施异常（系统没坏），是**业务语义边界**——组合本身不在登记里。对用户的正确回复是解释限制并给替代：

> “退款金额率当前支持按时间、渠道、地区查看。要看品类的话，净销售额支持按品类分组。”

而不是“系统出问题了”。

&emsp;&emsp;**第四步　收边界：别把结论扩大。** 拒绝绑在“指标 × 维度”的**组合**上，不单独绑在指标或维度上。看完整的组合矩阵（前两个拒绝码的含义：`METRIC_DIMENSION_UNSUPPORTED` 指标不支持该分组维度；`FILTER_FIELD_UNREGISTERED` 筛选字段未登记——筛选同样受该指标 dimensions 清单约束）：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-5　指标与维度组合矩阵</font></p>

<div class="center">

| 组合 | 结果 | 为什么 |
| :--- | :---: | :--- |
| 退款金额率 × 品类分组 | 拒绝 | product 不在它的 dimensions 清单 |
| 退款金额率 × 渠道分组 | 通过 | channel 在清单里 |
| 退款金额率 × 品类当筛选 | 拒绝 | 筛选同样受该指标 dimensions 约束 |
| 退款金额率 × 渠道筛选（写字段名 channel_name）+ 地区分组 | 通过 | 字段名在清单展开范围内 |
| 退款金额率 × 渠道筛选（写维度 ID channel）+ 品类分组 | 拒绝 | 双重违规：筛选字段名不合法 + 品类分组不在清单 |
| 净销售额 × 品类分组 | 通过 | net_sales 的清单里有 product |
| 支付订单数 × 品类分组 | 通过 | paid_orders 的清单里也有 product |

</div>

&emsp;&emsp;所以“退款分析不能用品类”是错误结论——品类这个维度本身没被封杀，换一个指标就合法。判断时永远查“这个指标的 dimensions 清单”，不要凭一次拒绝做归纳。

&emsp;&emsp;上面矩阵的七个组合，一段代码全部现场验证：

In [14]:
# 4.2：语义边界七组合现场验证（IntentValidator 逐个判定）
from hermes_analytics.nl2sql.intent.service import (
    IntentValidator, IntentValidationError,
)
from hermes_analytics.nl2sql.intent.models import AnalysisIntent, TimeRange
from datetime import date

validator = IntentValidator()

def try_submit(label, metric, group_by, filters=()):
    """提交一个指标组合并打印判定。

    Args:
        label: 教学输出标签。
        metric: 已登记指标 ID。
        group_by: 分组维度 ID 元组。
        filters: 可选筛选条件元组。

    Returns:
        None。
    """
    try:
        validator.validate(
            AnalysisIntent(
                metric=metric, group_by=group_by, filters=filters,
                time_range=TimeRange(start=date(2026, 6, 1), end=date(2026, 6, 30)),
                time_grain=None, comparison="none", ranking=None,
            ),
            catalog,  # 对照领域包登记判定
        )
        print(f"{label} -> VALID")
    except IntentValidationError as e:
        print(f"{label} -> {e.code} ({e.field})")

try_submit("退款金额率 x 品类分组     ", "refund_amount_rate", ("product",))
try_submit("退款金额率 x 渠道分组     ", "refund_amount_rate", ("channel",))
try_submit("退款金额率 x 品类当筛选   ", "refund_amount_rate", ("channel",),
           ({"field": "product", "operator": "eq", "value": "x"},))
try_submit("退款金额率 x 渠道名筛选+地区", "refund_amount_rate", ("region",),
           ({"field": "channel_name", "operator": "eq", "value": "tmall"},))
try_submit("退款金额率 x 渠道ID筛选+品类", "refund_amount_rate", ("product",),
           ({"field": "channel", "operator": "eq", "value": "tmall"},))
try_submit("净销售额   x 品类分组     ", "net_sales", ("product",))
try_submit("支付订单数 x 品类分组     ", "paid_orders", ("product",))

# 判定依据：两个指标的 dimensions 登记清单
for mid in ("refund_amount_rate", "net_sales"):
    m = next(x for x in catalog.metrics if x.id == mid)
    print(f"{mid}: {m.dimensions}")

退款金额率 x 品类分组      -> METRIC_DIMENSION_UNSUPPORTED (group_by)
退款金额率 x 渠道分组      -> VALID
退款金额率 x 品类当筛选    -> FILTER_FIELD_UNREGISTERED (filters)
退款金额率 x 渠道名筛选+地区 -> VALID
退款金额率 x 渠道ID筛选+品类 -> METRIC_DIMENSION_UNSUPPORTED (group_by)
净销售额   x 品类分组      -> VALID
支付订单数 x 品类分组      -> VALID
refund_amount_rate: ('time', 'channel', 'region')
net_sales: ('time', 'channel', 'product', 'region')


```text
# 码表：METRIC_DIMENSION_UNSUPPORTED=指标不支持该分组维度；FILTER_FIELD_UNREGISTERED=筛选字段未登记
退款金额率 x 品类分组     -> METRIC_DIMENSION_UNSUPPORTED (group_by)
退款金额率 x 渠道分组     -> VALID
退款金额率 x 品类当筛选   -> FILTER_FIELD_UNREGISTERED (filters)
退款金额率 x 渠道名筛选+地区 -> VALID
退款金额率 x 渠道ID筛选+品类 -> METRIC_DIMENSION_UNSUPPORTED (group_by)
净销售额   x 品类分组     -> VALID
支付订单数 x 品类分组     -> VALID
refund_amount_rate: ('time', 'channel', 'region')
net_sales: ('time', 'channel', 'product', 'region')
```

&emsp;&emsp;七行结果和上面的矩阵逐行对应，最后两行就是判定的登记依据——学员拿到任何“被拒绝”的组合，都可以用同样的办法先查这行清单再下结论。

&emsp;&emsp;判断边界可以固定成三分法：缺字段 → 发起澄清；指标 × 维度不支持 → 解释并给替代；基础设施异常 → 进入系统排障。模型和程序的分工是：程序负责机器可读判定，模型负责把判定转成用户语言。

&emsp;&emsp;下面这张机制图把三种情况分开，避免你把业务语义边界当成数据库故障。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172938091.png" alt="缺口、语义边界、基础设施异常三分法机制图" width="85%"></div>

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 4-6　三种情况的分辨</font></p>

<div class="center">

| 现象 | 类型 | 正确动作 |
| :---: | :---: | :---: |
| “哪个渠道表现最好”且 `gaps` 指向指标/时间 | 意图缺口 | 按 `gaps` 补齐，不预设轮数 |
| “列出表现最好的渠道”且缺少排名数量 | Top N 缺口 | 补充具体数量 |
| “退款金额率按品类” | 语义边界 | 解释业务限制，给合法替代 |
| 后端 5xx、数据库连接失败 | 基础设施异常 | 进入系统排障 |

</div>

### 4.3 先判断，再看参考答案

<details>
<summary>判断一：“2026 年 6 月各品类退款金额率”应该继续执行、继续澄清，还是进入系统排障？</summary>

这是指标 × 结果维度的业务语义边界。当前退款金额率支持的结果维度不含 product/品类，应解释限制并给出合法替代维度；不能把它归为数据库故障，也不要扩大成“所有退款分析都不能使用品类”。

</details>

<details>
<summary>判断二：“看看表现”两个字，模型下一步会做什么？</summary>

意图缺口。没有指标、没有时间，“表现”无法映射到任何已登记指标；预期 `gaps` 出现 `METRIC_REQUIRED` + `TIME_RANGE_REQUIRED`，模型发起澄清而不是猜一个口径。可用机制卡①第一段代码把 `metric` 和 `time_range` 改成 `None` 自行跑一遍确认。

</details>

<details>
<summary>判断三：同样的问法昨天能跑、今天报“数据库连接失败”，属于哪一类？</summary>

基础设施异常。问法没变、登记没变，变的是运行环境；按三分法应进入系统排障（检查连接、服务状态），而不是回头改问法或怀疑语义边界。

</details>

&emsp;&emsp;**本章自测**：给定任意 `gaps`，你能补齐指标、时间或 Top N，并用三分法判断属于缺口澄清、业务语义边界还是系统排障。实际澄清轮数与措辞以运行结果为准。

&emsp;&emsp;**机制卡①　缺口检测是一段确定性程序的机器码**：`ClarificationDetector.find_gaps`（[`nl2sql/intent/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/intent/service.py)，当前约 242 行起）吃一份草稿 Intent，逐字段判定缺失，只回机器可读缺口码，不写中文问句。模型永远收不到「帮你猜一个时间或指标」的授权。

&emsp;&emsp;下面直接运行这段确定性代码，观察三种场景的缺口检测结果：

In [15]:
# 机制卡①：ClarificationDetector.find_gaps 现场运行
from hermes_analytics.nl2sql.intent.service import ClarificationDetector

detector = ClarificationDetector()

# 场景1：缺指标 + 缺时间
gaps1 = detector.find_gaps(
    {"metric": None, "time_range": None, "group_by": [],  # 草稿 Intent 字典
     "filters": [], "comparison": "none", "ranking": None},
    catalog,  # 领域包
)
print(f"缺指标+缺时间 -> {[g.code for g in gaps1]}")

# 场景2：有指标有时间，缺 Top N 数量
gaps2 = detector.find_gaps(
    {"metric": "paid_orders",
     "time_range": {"start": "2026-06-01", "end": "2026-06-30"},
     "group_by": ["channel"], "filters": [], "comparison": "none",
     "ranking": {"by": "value", "direction": "desc", "limit": None}},
    catalog,
)
print(f"缺TopN -> {[g.code for g in gaps2]}")

# 场景3：完整 Intent，无缺口
gaps3 = detector.find_gaps(
    {"metric": "paid_orders",
     "time_range": {"start": "2026-06-01", "end": "2026-06-30"},
     "group_by": ["channel"], "filters": [], "comparison": "none",
     "ranking": None},
    catalog,
)
print(f"完整Intent -> {[g.code for g in gaps3] if gaps3 else '(无缺口)'}")

缺指标+缺时间 -> ['METRIC_REQUIRED', 'TIME_RANGE_REQUIRED']
缺TopN -> ['RANKING_LIMIT_REQUIRED']
完整Intent -> (无缺口)


```text
# 输出里的缺口码含义：METRIC_REQUIRED=缺指标 TIME_RANGE_REQUIRED=缺时间 RANKING_LIMIT_REQUIRED=缺排名数量
缺指标+缺时间 -> ['METRIC_REQUIRED', 'TIME_RANGE_REQUIRED']
缺TopN -> ['RANKING_LIMIT_REQUIRED']
完整Intent -> (无缺口)
```

&emsp;&emsp;上面三个场景直调了 `find_gaps`；生产链路走的是同一检测器的另一个入口 `validate_or_find_gaps_detailed`——`bridge.py` 用它先归一化、再检测、再分类。用 3.3 的真实提交格式再跑一次，看它在生产入口下如何分诊：

In [16]:
# 机制卡① 续：生产入口的分诊（bridge.py 同款调用）
intent, gaps, errors = detector.validate_or_find_gaps_detailed(
    {"metric": "paid_orders", "time_range": None,  # 模型没填时间，其余齐备
     "time_grain": None, "group_by": ["channel"],
     "filters": [], "comparison": "none", "ranking": None},
    catalog,
)
print(f"intent: {intent}")                       # None：没形成完整意图
print(f"gaps:   {[g.code for g in gaps]}")        # 退回模型的缺口码
print(f"errors: {[e.code for e in errors]}")      # 诊断明细（入日志）


intent: None
gaps:   ['TIME_RANGE_REQUIRED']
errors: ['TIME_RANGE_REQUIRED']


```text
# intent=None 表示未形成完整意图；gaps 是退回模型的缺口清单；errors 是入日志的诊断明细
intent: None
gaps:   ['TIME_RANGE_REQUIRED']
errors: ['TIME_RANGE_REQUIRED']
```

&emsp;&emsp;对照第 4.1 节：页面上的澄清问句会变，但「缺口码由谁产生、缺什么才算缺」不变——它只来自这一段确定性代码，不在模型判断里。3.3 第 2 次调用里模型主动提交 `kind=clarification`，是模型在拿到这类反馈（或预判到会有这类反馈）后的主动动作；Core 的检测始终是机器可读的那一半。

&emsp;&emsp;下一章问题：当分析意图已经完整后，连续追问时哪些条件应该保留？

## <center>第五章：会话与上下文</center>

&emsp;&emsp;本章我们承接“补全意图”进入“管理上下文”。上一轮已经形成“2026 年 6 月各渠道净销售额”，现在用户说“只看华东地区呢？”、“支付订单数呢？”、“重新分析”。每一轮都要重新形成完整分析意图，但并不是所有条件都从零开始。

&emsp;&emsp;本章核心问题：上一轮的哪些条件应该保留，哪些条件必须替换、丢弃或清空？

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-1　常见误解与真实机制</font></p>

<div class="center">

| 常见误解 | 真实机制 | 判断边界 |
| :---: | :---: | :---: |
| 上下文会自动拼接上一轮 JSON | 模型在本轮重新形成完整 Intent | 每轮仍要提交完整分析意图 |
| 字段存在就一定继续适用 | 字段要符合当前分析对象和业务语义 | 旧对象专属筛选可能失效 |
| 切换分析对象就要清空所有条件 | 时间等仍适用条件可以保留 | 只丢弃不再有效的筛选 |
| “重新分析”会删除历史会话 | 重置清空当前继承上下文，历史 Turn 保留 | 当前上下文和历史展示是两件事 |

</div>

&emsp;&emsp;**源码锚点**：完整 Intent 与上下文提示看 [`prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py) 和 [`bridge.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_planning/bridge.py)；多轮追问/继承样例看 [`prompt_bench.py`](HermesAnalytics第二版/scripts/quality/prompt_bench.py) 与 [`classroom_bench.py`](HermesAnalytics第二版/scripts/quality/classroom_bench.py)；重置存储状态看 [`control_repository.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/postgres/control_repository.py)，历史交互看 [`AnalystConversationWorkspace.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx)。

&emsp;&emsp;**口语类比与反例**：每一轮都像提交新版分析申请单，而不是贴一张零散便签。反例是已经筛选“抖音”后再问“哪个渠道最好”，却仍机械保留抖音筛选。

### 5.1 多轮追问的机制：谁来补全，凭什么补得对

&emsp;&emsp;先把问题摆实。上一轮已经形成"2026 年 6 月各渠道净销售额"，现在用户说"只看华东地区呢？"——这句话没有指标、没有时间、没有分组，它本身**进不了确定性链**：QueryPlanner 要七字段齐全才能规划，缺一个都走不通。但用户显然预期系统能接着聊。这一节就回答：残缺的追问凭什么接得住，谁把它补全，补的依据是什么。

&emsp;&emsp;**第一步　补全只能发生在模型侧。** 第四章立过架构铁律：程序不从自然语言做推断。"只看华东呢？"里的"呢"指向上一轮，这个指代关系只有理解语义才能解开——所以补全这件事天然落在模型头上，程序不许碰。Core 期待收到的是一份字段齐全的提交；模型偶尔也会交残缺卷子——缺口检测当场拦下，带着缺口码退回重写（第四章机制卡①就是这段代码）。所以补全是模型的本职，程序只兜底拦截。

&emsp;&emsp;**第二步　模型补全的依据是注入的上一轮上下文。** 程序不推断，但程序**供料**。每轮规划提示词里都注入了 `turn_context.analysis_context.last_intent`——上一轮已确认的完整 Intent。模型拿到的指令原文（[`prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py) 多轮追问段）：

> turn_context.analysis_context.last_intent 是上一轮已确认的意图。你是唯一掌握上下文的一方，Core 不会替你从上一轮取任何值，本轮需要什么就自己写进本轮 Intent。

&emsp;&emsp;注意这两句话的分工：**上一轮通过提示词告知，本轮由模型重写**。继承是模型参照着写，不是程序把旧字段拼进新 JSON——`bridge.py` 的 `_normalize_intent` 归一化函数注释写得很直白："本轮要查什么由 Hermes 判断并写进 Intent……Core 只裁决能不能查：缺失或非法一律走校验与澄清，**绝不从上一轮取值回填**。"

&emsp;&emsp;**第三步　为什么不让程序拼接——看一个陷阱。** 假如让程序机械拼接：上一轮 Intent 带着筛选 `channel_name="douyin"`（只看抖音），这一轮用户问"哪个渠道净销售额最高？"——拼接器会把抖音筛选原样带进本轮，生成"在抖音内部比高低"的 Intent。这个组合拿去校验**完全合法**（实测 VALID），SQL 正常出结果，页面显示"最高的是抖音"——**错误悄无声息地发生了**。用户明明在问全渠道比较，答案却被旧筛选锁死。判断"旧筛选还适不适用"是语义判断，程序做不了，这正是继承必须交给模型的原因。

&emsp;&emsp;**第四步　继承规则写进了提示词，四类行为各有明确指令。** [`prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py) 的多轮追问段把常见连续追问归成四类，每类直接写明保留什么丢什么（下面表格的右两列就来自这段原文）：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-2　四类追问行为与继承规则（源自 prompt.py 多轮追问段）</font></p>

<div class="center">

| 用户行为 | 典型话术 | 应保留 | 应丢弃/替换 |
| :---: | :---: | :---: | :---: |
| 只换指标 | "那另一个指标呢" | 时间、分组、筛选照写 | 替换指标 |
| 追加限定 | "只看这个成员" | 原有口径之上追加 | 无丢弃 |
| 切换分析对象 | "哪个分组成员表现最好" | 时间范围照写 | 丢弃单一成员筛选 |
| 明确重置 | "重新分析" | 历史展示保留 | 一律不继承 |

</div>

&emsp;&emsp;表中话术是提示词原文的归类用语；主任务里的"支付订单数呢？""只看华东地区呢？"分别落到第一、二行——归类看语义不看字面，同是追问"呢"，问的是换指标还是加限定由意思决定。

&emsp;&emsp;第三行的丢弃规则就是第二步陷阱的官方答案——提示词原文："换了分析对象（'哪个分组成员表现最好'）→ 丢掉上一轮针对单一成员的筛选，**否则会在只剩一个成员的结果里回答'哪个最好'**；但时间范围照写，换分析对象不等于换时间。"还有一条兜底："拿不准某个筛选是否还成立，宁可不写让 Core 走澄清，也不要默认沿用。"

&emsp;&emsp;**第五步　继承的载体与关闭开关。** 模型每轮参照的 `last_intent` 存在 `current_context` 里，与之并列的还有 `last_query_id`（上一轮查询的来源指针）。"重新分析"按钮清空的正是这两个字段——切断注入，下一轮模型拿不到上一轮，自然只能从用户新问题本身出发。历史轮次（`conversation_turns`）属于展示与审计层，与继承无关，重置一概不动。

&emsp;&emsp;把五步合起来看，"上下文"这个词其实指了三层东西，别混：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-3　"上下文"的三层状态（重置只动第二层）</font></p>

<div class="center">

| 层 | 存什么 | 谁写谁读 | 重置时 |
| :---: | :---: | :---: | :---: |
| 历史轮次 conversation_turns | 每轮问答记录 | 展示与审计用 | 不删 |
| 继承入口 current_context | `last_intent` + `last_query_id` | 提示词注入给模型 | **清空两字段** |
| 本轮完整 Intent | 模型重写的七字段 | Core 校验 | 照常提交 |

</div>

&emsp;&emsp;"重新分析"这条链路需要从前端到数据库完整观察一次。前端调用网关重置上下文；后端路由接受请求；应用服务调用仓储；仓储只更新 `current_context` 的 `last_intent` 和 `last_query_id`，不删除 `conversation_turns`。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-4　"重新分析"完整调用链</font></p>

<div class="center">

| 环节 | 精确符号 | 当前行段 | 作用 |
| :---: | :---: | :---: | :---: |
| 前端按钮 | `AnalystConversationWorkspace.resetContext` | `frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx` 当前约 1527–1556 行 | 调用网关重置 |
| 前端网关 | `HttpAnalyticsGateway.resetAnalysisContext` | [`HttpAnalyticsGateway.ts`](HermesAnalytics第二版/frontend/src/shared/gateway/HttpAnalyticsGateway.ts) 当前约 733 行起 | 发送重置请求 |
| API 路由 | `api/routes/sessions.py reset_context` | `backend/src/hermes_analytics/api/routes/sessions.py` 当前约 186–216 行 | 校验身份后调用应用服务 |
| 应用服务 | `application/sessions/service.py reset_context` | `backend/src/hermes_analytics/application/sessions/service.py` 当前约 148–171 行 | 组装重置命令 |
| 数据仓储 | `PostgresControlRepository.reset_analysis_context` | `backend/src/hermes_analytics/infrastructure/postgres/control_repository.py` 当前约 374–442 行 | 清空 `last_intent`/`last_query_id`，历史保留 |

</div>

&emsp;&emsp;用主任务举例，在同一会话内按以下顺序观察。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-5　同一会话中的口径变化</font></p>

<div class="center">

| 轮次 | 指标 | 时间 | 分组 | 地区 | 继承判断 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 完整初问 | 净销售额 | 2026 年 6 月 | 渠道 | 不限定 | — |
| 只看华东 | 净销售额 | 2026 年 6 月 | 渠道 | 华东 | 追加限定：照抄上一轮再追加地区 |
| 支付订单数呢 | 支付订单数 | 2026 年 6 月 | 渠道按用户预期保留 | 华东按用户预期保留 | 只换指标：口径照写、指标替换 |
| 重新分析 | — | — | — | — | 继承入口清空，历史仍保留 |

</div>

&emsp;&emsp;注意上表的"按用户预期保留"——这是验收口径，不是程序保证：程序只负责注入上一轮和校验本轮，保留不保留是模型照四类规则写的，写错了会被校验或运行结果暴露。不承诺模型每次都按相同措辞或轮数完成继承。

&emsp;&emsp;下面这张机制图说明每轮都会重新形成完整 Intent；字段按需保留、新增或替换，重置停止继承但保留历史。它不是运行截图。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819173033128.png" alt="每轮完整 Intent、字段保留新增替换、reset 停止继承、历史保留机制图" width="85%"></div>

### 5.2 重置边界：一个精确的关闭开关

&emsp;&emsp;"重新分析"在页面上只是一个按钮，落到数据库是一条 UPDATE 语句。这条语句只做一件事：

```sql
-- control_repository.py reset_analysis_context 的核心更新（约 419 行起）
UPDATE system.analysis_sessions
SET current_context = jsonb_build_object('last_intent', NULL, 'last_query_id', NULL),
    version = version + 1
```

&emsp;&emsp;`current_context` 整体被替换成一个只有两个 null 字段的对象——继承入口关闭。没有 DELETE，`conversation_turns` 一行不删；也没有任何触发旧 Query 重跑的语句，后续新问题才会形成新 Query。三句"不做什么"由此可以精确回答：**不删历史、不重跑旧查询、不只清一个模糊的"上下文"——清的是两个具名字段**。

&emsp;&emsp;这条 UPDATE 周围还有三道保护，说明重置是受控动作而不是随手清空：执行前对会话行加 `FOR UPDATE` 锁并校验 `expected_version`（乐观并发，防止两次点击互相覆盖）；幂等键 `session.reset:{owner}:{session}:{request_id}`（同一请求重放直接返回上次结果）；完成后写入审计表。前端点按钮 → 网关 → 路由 → 服务 → 仓储的完整链路见表 5-4。

&emsp;&emsp;下面是"运行截图｜Compose 项目 hermes-analytics-final"：页面显示"上下文已清空，历史对话仍保留"。它只证明本次重置后的页面提示，不承诺所有版本的措辞完全相同。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172943601.png" alt="运行截图：重置后上下文已清空、历史对话仍保留" width="85%"></div>

&emsp;&emsp;重置之后怎么验收？三个可观察点，每个都有明确证据：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 5-6　重置后的可观察验收卡</font></p>

<div class="center">

| 检查对象 | 验收条件 | 证据类型 |
| :---: | :---: | :---: |
| 当前上下文 | `last_intent` 清空 | 源码机制 + 运行截图 |
| 来源关联 | `last_query_id` 清空 | 源码机制 + 运行截图 |
| 历史记录 | 既有 Turn 仍可见 | 交互机制 + 运行截图 |

</div>

&emsp;&emsp;验收卡背后是三层状态的分工（表 5-3）：重置只动第二层继承入口，第一层历史和第三层本轮提交都不受影响。学员如果问"历史还在为什么说不继承了"，答案就是这两层本来就管不同的事——历史给人看，继承入口给模型看。

### 5.3 先判断，再看参考答案

<details>
<summary>已经筛选“抖音”后追问“哪个渠道净销售额最高”，旧渠道筛选和时间范围分别怎样处理？</summary>

丢弃只针对抖音的渠道筛选；保留仍适用的时间范围；本轮重新提交完整分析意图。这正是 5.1 第三步讲过的陷阱反过来考：机械拼接会答出“最高的是抖音”，模型必须丢掉单一成员筛选才能正确回答全渠道比较——prompt.py 对这一类的指令是“丢掉上一轮针对单一成员的筛选，否则会在只剩一个成员的结果里回答‘哪个最好’”。

</details>

&emsp;&emsp;**本章自测**：你能判断主任务中的追加限定、只换指标和重置，并能用独立变体解释切换分析对象；重置后按验收卡核对两个字段清空、历史 Turn 保留。

&emsp;&emsp;**机制卡②　每轮都是一次完整的意图重提**：程序不把「上一轮缺的字段」或「上一轮的结果」偷偷填回本轮。`PlanningToolBridge.submit_query_plan`（[`bridge.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_planning/bridge.py)，约 164 行起）只收本轮模型的七字段提交，再交 `IntentValidator.validate` 逐字段重新校验；上一轮只通过提示词告知上下文，程序本身不回填值。


&emsp;&emsp;下面直接运行 `IntentValidator.validate`，观察合法 Intent 和非法指标的不同结果：

In [ ]:
# 机制卡②：IntentValidator.validate 现场运行
from hermes_analytics.nl2sql.intent.service import IntentValidator, IntentValidationError
from hermes_analytics.nl2sql.intent.models import AnalysisIntent, TimeRange
from datetime import date

validator = IntentValidator()  # 无参构造

# 合法 Intent：校验通过，返回 AnalysisIntent
try:
    intent_ok = AnalysisIntent(
        metric="paid_orders",
        time_range=TimeRange(start=date(2026, 6, 1), end=date(2026, 6, 30)),
        time_grain=None, group_by=("channel",), filters=(),
        comparison="none", ranking=None,
    )
    result = validator.validate(intent_ok, catalog)  # 成功返回 AnalysisIntent
    print(f"合法Intent -> VALID (metric={result.metric})")
except IntentValidationError as e:
    print(f"合法Intent -> INVALID: {e.code}")

# 非法指标：校验失败，抛 IntentValidationError
try:
    intent_bad = AnalysisIntent(
        metric="nonexistent_metric",  # 领域包中未登记的指标
        time_range=TimeRange(start=date(2026, 6, 1), end=date(2026, 6, 30)),
        time_grain=None, group_by=("channel",), filters=(),
        comparison="none", ranking=None,
    )
    validator.validate(intent_bad, catalog)
except IntentValidationError as e:
    print(f"非法指标 -> INVALID: {e.code} ({e.field})")

```text
合法Intent -> VALID (metric=paid_orders)
非法指标 -> INVALID: METRIC_UNREGISTERED (metric)
```

&emsp;&emsp;这段代码是对 5.1 五步主线的收口：程序侧没有「拼接」这个动作——校验的对象始终是本轮这份完整提交，合法它就过、非法它就拒，跟历史上提交过什么毫无关系。抖音陷阱之所以不会发生，不是因为程序会识别陷阱，而是因为程序根本没有能力把旧筛选带进来；带不带，只在模型一支笔上。重置侧的对应事实：只清 `last_intent`/`last_query_id` 两字段，`conversation_turns` 不删（见 5.2 节）。判断标准始终是「本轮最终形成的那份完整 Intent」，不是「系统应该记得什么」。

&emsp;&emsp;下一章问题：新问题停在人工确认点后，怎样进入 SQL 工作台做受控复核？

## <center>第六章：SQL 工作台受控复核</center>

&emsp;&emsp;本章我们把受控复核拆成四段状态接力：入口资格 → 参数草稿 → 重新校验与执行 → 受控回流。核心问题是：两条 SQL 都能运行时，为什么只有具备来源与血缘的一条可以返回 Hermes？

&emsp;&emsp;这里先回顾开篇的 Intent、Core、gaps、execution，再补三个与 SQL 联动和证据定位直接相关的词。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-1　本精简术语卡</font></p>

<div class="center">

| 术语 | 一句话定义 |
| :---: | :---: |
| Seed | 从待确认 Query 派生联动 SQL 时使用的服务端来源凭证 |
| validation | 对当前 SQL 来源和参数草稿做出的校验结果 |
| FactRef | 服务端生成并允许模型选择的结果行、列、值坐标 |

</div>

> **SQL 联动证据状态**：主任务已覆盖待确认、可编辑 Seed、改参、VALID、SUCCEEDED 与受控回流；结果解读失败界面仍待运行验证。

&emsp;&emsp;先拆误解：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-2　常见误解与真实机制</font></p>

<div class="center">

| 常见误解 | 真实机制 | 判断边界 |
| :---: | :---: | :---: |
| 自由 SQL 执行成功就能返回 Hermes | 自由 SQL 没有分析会话来源与血缘 | 执行成功不等于具备回流资格 |
| 会话联动 SQL 可以自由改结构 | SQL 结构只读，只能修改批准参数 | 无批准参数时保留原值 |
| 前端显示可编辑就是最终裁决 | 后端仍校验参数、版本和来源 | 前端是操作提示，后端是最终边界 |
| 参数变化后仍能沿用旧 ALLOW | 旧 validation 与旧 execution 对当前参数草稿的提交资格失效 | 历史 execution 保留，但新草稿必须产生新 execution |

</div>

&emsp;&emsp;**源码锚点**：工作台契约看 [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py)，前端只读与参数区看 [`SqlWorkbench.tsx`](HermesAnalytics第二版/frontend/src/features/workbench/components/SqlWorkbench.tsx)，页面装配看 [`SqlWorkbenchPage.tsx`](HermesAnalytics第二版/frontend/src/features/workbench/pages/SqlWorkbenchPage.tsx)，后端路由看 [`workbench.py`](HermesAnalytics第二版/backend/src/hermes_analytics/api/routes/workbench.py)，回流看 [`turns.py`](HermesAnalytics第二版/backend/src/hermes_analytics/api/routes/turns.py)。

&emsp;&emsp;**口语类比与反例**：自由 SQL 像个人草稿，联动 SQL 像带原始单据编号的正式复核单。反例是改了日期后继续使用旧 ALLOW，或页面没有批准参数时强改 SQL 结构。

### 6.1 状态一：入口资格

&emsp;&emsp;先用一张前置卡恢复第一节结论：自由 SQL 可以独立校验和执行，但不能返回 Hermes；会话联动 SQL 必须来自待确认 Query，并接受 Seed、参数权限和服务端血缘约束。本节不再重复它们的基础定义，而是观察参数变化后的状态接力。

```text
自由 SQL：独立编辑 → 校验 → 执行 → 结果留在工作台 → 不返回 Hermes
会话联动 SQL：待确认 Query + 有效 Seed → 批准参数 → 新 validation → 新 execution → 服务端复核 → 返回 Hermes
```

&emsp;&emsp;先把本章最容易混淆的一件事说死：**"能不能执行"和"能不能回流"是两道不同的关卡，管的不是同一件事。**

&emsp;&emsp;第一道关卡是**安全闸门**——判断这条 SQL 能不能碰数据库。任何 SQL 在执行前都要过白名单：只读单条、对象边界、无星号、函数白名单（机制卡③ 会现场运行这套判定）。这道关卡自由 SQL 和联动 SQL **都要过**，谁也没有豁免。它回答的问题是"这条 SQL 危不危险"。

&emsp;&emsp;第二道关卡是**回流资格**——判断这次执行的结果能不能被 Hermes 采信为会话证据。这道关卡看的不是 SQL 文本，而是两样自由 SQL 天生没有的东西：**血缘**（`source_query_id`——这次执行从哪个分析问题派生，链条能否追溯）和 **plan_hash 冻结**（SQL、参数、计划、领域包哈希、语义哈希整体入哈希——结果与口径是否可复现可复核）。它回答的问题是"这个结果可不可信、归哪个问题所有"。

&emsp;&emsp;所以"自由 SQL 执行成功也不能返回 Hermes"的真正答案：**不是安全检查没过（它同样过了安检），而是拿不出血缘和冻结凭证**。就像机场安检和海关：安检过了只说明你不危险，能不能入境看的是护照和签证。全章后面三段状态接力（参数草稿 → 重新校验与执行 → 受控回流），讲的都是"护照和签证"这条线的办理流程。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-3　SQL 双路径前置卡</font></p>

<div class="center">

| 对比项 | 自由 SQL | 会话联动 SQL |
| :---: | :---: | :---: |
| 来源标识 | `ANALYST_AD_HOC` | `CONVERSATION_DERIVED` |
| 入口 | SQL 工作台独立新建 | Query 处于 `AWAITING_EXECUTION_CONFIRMATION` 且 Seed 有效 |
| SQL 编辑 | 可以编辑，仍需校验 | SQL 结构只读 |
| 参数调整 | 分析师自行编写 | 只改页面批准的可编辑参数 |
| 安全闸门（WBSQL 白名单） | 同样要过，六项判定 | 编译期已过 SQLP 四项，工作台期再复核 |
| 返回 Hermes | 不可以——缺血缘与冻结凭证 | 符合条件时返回分析工作台 |
| 第二节重点 | 不展开基础操作 | 参数草稿变化后重新取得提交资格 |

</div>

&emsp;&emsp;下面的机制图把两道关卡画进同一张图：安全闸门两条路径都要过；回流资格只有联动路径能办——`CONVERSATION_DERIVED` 且 Seed 有效只是必要条件，后续条件在 6.3、6.4 逐步补齐。它不是运行截图。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172941415.png" alt="自由 SQL 与会话联动 SQL 双路径机制图" width="85%"></div>

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：Query 已冻结，页面等待确认，系统尚未访问数据库。它证明“等待确认”是实际可见状态，不是模拟。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172945684.png" alt="运行截图：等待确认 Query，SQL 已冻结尚未访问数据库" width="85%"></div>

&emsp;&emsp;课堂先检查三项必要条件：来源 Query 存在、状态为 `AWAITING_EXECUTION_CONFIRMATION`、Seed 有效——第二道关卡的入口检查，对应总纲里"血缘"那本护照的初次验证。三项之外，完整回流还要经过 validation、execution 与血缘复核；已经完成的 Query、过期 Seed 或无 Seed 都不能进入联动路径。

&emsp;&emsp;入口资格成立后，才能从“待确认 Query”进入 `CONVERSATION_DERIVED` 联动 SQL。此后 6.2 到 6.4 的三段状态接力，全程都在第二道关卡内推进；安全闸门只在每个需要执行的时刻静默复查（6.3 重新校验时会再次运行）。

### 6.2 状态二：参数草稿

&emsp;&emsp;进入后，页面显示 SQL 结构、来源信息和参数区。SQL 结构只读；你只能修改明确批准的参数。若没有批准的可编辑参数，就保留原值，不强行改 SQL。参数草稿只是准备状态，还不具备执行资格。

&emsp;&emsp;时间参数采用半开区间：`period_start ≤ 时间 < period_end`，即开始日期包含在范围内，结束日期不包含。例如 `2026-05-01` 到 `2026-06-01` 表示 5 月 1 日至 5 月 31 日的数据；界面回流时会把结束日期减一天，显示为“2026-05-01 至 2026-05-31”，方便你直接读。

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：来源为 `CONVERSATION_DERIVED`，SQL 结构只读，参数面板显示 `period_start`、`period_end` 与 `region_ids`。它证明本次演示可编辑的是时间参数；不要把“所有日期都可改”写成规律，参数是否可改由冻结 contract 决定。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172946350.png" alt="运行截图：CONVERSATION_DERIVED 联动 SQL 只读结构与可编辑参数面板" width="85%"></div>

&emsp;&emsp;三类典型非法参数是越界日期、反向日期和非法枚举值。例如结束日期早于开始日期，或地区值不在领域包枚举内，都会触发受控拒绝；这是参数边界，不是系统故障。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-4　SQL 工作台六个功能区观察卡</font></p>

<div class="center">

| 功能区 | 页面看什么 | 系统做什么 | 判断边界 | 源码定位 |
| :---: | :---: | :---: | :---: | :---: |
| 来源与模式 | 来源是 `CONVERSATION_DERIVED` 还是自由 SQL | 按来源决定返回资格 | 自由 SQL 执行成功也不回流 | `WorkbenchValidationRequest.validate_variant`，`api/schemas/workbench.py` 当前约 70–85 行 |
| 只读 SQL | SQL 编辑器只读 | 结构来自 Seed，不接收用户改 SQL | 联动模式不能改 SQL 结构 | `SqlWorkbench`，`frontend/src/features/workbench/components/SqlWorkbench.tsx` 当前约 83–230 行，联动只读属性约 629 行 |
| 命名参数 | 可编辑参数面板 | 后端从 Seed 契约投影可修改字段 | 只有批准参数可改 | `NamedParametersPanel`，`frontend/src/features/workbench/components/NamedParametersPanel.tsx` 当前约 76 行起；后端 `WorkbenchService._derived_material` 最终裁决 |
| 校验结果 | `VALID/ALLOW` 或拒绝信息 | 重新执行 Workbench Policy | 参数变化后旧校验失效 | `WorkbenchService.validate`，`application/workbench/service.py` 当前约 58–168 行 |
| 执行结果 | `SUCCEEDED` 与 execution | 绑定 `validation_id + plan_hash` | 新 execution 才代表当前草稿 | `WorkbenchService.execute`，`application/workbench/service.py` 当前约 376–523 行 |
| 返回分析工作区 | 回流按钮与来源引用 | 只提交 `execution_id + source_query_id` | 服务端复核血缘后才交给 Hermes | `resume_from_workbench`，`api/routes/turns.py` 当前约 203–243 行 |

</div>

&emsp;&emsp;参数不是前端随便放的输入框，而是由服务端契约决定值域。四类约束分别来自 editable 白名单、治理目录、数据覆盖区间和跨参数关系；其中后三类对自由 SQL 的自助提交同样生效（源码里两条路径汇聚到同一处检查），只有 editable 白名单是 Seed 契约专属。前端控件只负责提示，后端是最终裁决者。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-5　四类参数约束</font></p>

<div class="center">

| 约束 | 含义 | 超出边界时 | 源码位置 |
| :---: | :---: | :---: | :---: |
| `editable` 白名单 | 只有 Seed 契约标记可改的参数才能提交 | `WORKBENCH_PARAMETER_FORBIDDEN` | `WorkbenchService._derived_material`，`application/workbench/service.py` 当前约 170–276 行 |
| `catalog_source` | 带治理目录来源的参数必须逐项落在目录 | `WORKBENCH_PARAMETER_OUT_OF_CATALOG` | `WorkbenchService._reject_values_outside_catalog`，`application/workbench/service.py` 当前约 280–306 行 |
| `bounds_source` | 日期参数不得超出数据覆盖区间 | `WORKBENCH_PARAMETER_OUT_OF_COVERAGE` | `WorkbenchService._reject_values_outside_data_coverage`，`application/workbench/service.py` 当前约 309–342 行 |
| `parameter_relations` | 被修改参数涉及跨参数关系时重新校验 | `WORKBENCH_PARAMETER_RELATION_VIOLATED` | `WorkbenchService._reject_relation_violations`，`application/workbench/service.py` 当前约 344–374 行 |

</div>

### 6.3 状态三：重新校验与执行

&emsp;&emsp;当批准参数发生变化时，旧 validation 以及旧 execution 相对于当前参数草稿的提交资格失效；历史 execution 记录仍保留。你需要用当前批准的 `parameter_values` 重新校验，随后生成与当前参数一致的新 execution。

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：把 `period_start`/`period_end` 从 2026-06-01~2026-07-01 改为 2026-05-01~2026-06-01，页面进入待校验状态。它证明参数变更后不能直接沿用旧校验。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172946470.png" alt="运行截图：参数改为 2026-05-01 至 2026-06-01，状态待校验" width="85%"></div>

&emsp;&emsp;参数变更后的状态接力是本节主线。前端的 UI 状态机依次走过 EDITING（改参即回到此态，旧结果清除）→ VALIDATING → VALID → EXECUTING → SUCCEEDED（九态中的五态，完整列表见前端 `workbenchState.ts`）；后端配套产出新 validation 和新 execution，回流由服务端复核放行。下面这张机制图标出这条链路。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172941451.png" alt="Seed 到编辑、校验、执行、服务端复核回流的状态接力机制图" width="85%"></div>

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：新草稿重新校验后得到 `VALID`/`ALLOW`。它只证明当前草稿取得了提交资格，不表示旧草稿或后续新草稿仍可沿用。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172947413.png" alt="运行截图：参数草稿重新校验后 VALID/ALLOW" width="85%"></div>

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：执行后状态为 `SUCCEEDED`，并产生新 execution。历史 execution 不删除，但当前草稿的提交资格只绑定新 execution。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172947860.png" alt="运行截图：新 execution 状态 SUCCEEDED" width="85%"></div>

&emsp;&emsp;这里的重点不是记住每个字段，而是判断状态接力是否完整：参数草稿变化 → 旧提交资格失效 → 新 validation → 新 execution。任何一步缺失，都不能把旧执行当作当前草稿的执行结果。

### 6.4 状态四：受控回流与实际口径

&emsp;&emsp;回流阶段只提交两个引用：`execution_id`（哪次执行）与 `source_query_id`（从哪个 Query 派生），不上传参数快照，也不上传结果行。服务端拿到这两个引用后复核三样东西——状态（执行是否 SUCCEEDED）、版本（会话有没有被并发操作过期）、血缘（这条执行是否真从那个 Query 派生）——复核通过，才把与新 execution 对应的证据交给 Hermes 解读。`expected_turn_version` 与 `client_request_id` 属于版本和幂等细节，放到课后源码核验，不占主流程记忆。

&emsp;&emsp;复核通过后，系统生成 `WORKBENCH_REVISION` 标记的 Query 和 `ANALYSIS_VERIFIED` 级别的快照。注意这一步**不重新执行 SQL**：快照直接复用工作台那次执行的结果行，服务端只校验结果指纹是否一致。换句话说，数据库只在执行那一步被碰过一次；回流做的是"验明正身 + 升级可信级别"，不再花第二次查询成本。

&emsp;&emsp;下面是“运行截图｜Compose 项目 hermes-analytics-final”：受控回流页面明确显示实际执行口径为 2026-05-01 至 2026-05-31，与原问题“2026 年 6 月”不同，并带 FactRef。它证明最终解释跟随实际执行参数。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172948272.png" alt="运行截图：受控回流显示实际执行口径 2026-05-01 至 2026-05-31 与 FactRef" width="85%"></div>

&emsp;&emsp;回流页面上最扎眼的就是那行口径："2026-05-01 至 2026-05-31"。用户当初问的是 2026 年 6 月，回来的答案却是 5 月——这不是系统错了，而是本章最重要的一条口径纪律在起作用。把三个阶段的时间口径摆在一起对照：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-6　原问题与实际执行口径对照</font></p>

<div class="center">

| 阶段 | 时间口径 | 解释要求 |
| :---: | :---: | :---: |
| 原问题 | 2026 年 6 月华东地区各渠道净销售额 | 只是进入工作台前的原始口径 |
| 工作台参数草稿 | 若获批改为 2026 年 5 月 | 必须重新校验并产生新 execution |
| 回流解释 | 2026 年 5 月 | 不得沿用原问题中的“2026 年 6 月”措辞 |

</div>

&emsp;&emsp;为什么解释必须跟着 execution 走、不能跟着原话走？因为整条链里真正碰过数据库的只有那次执行：原话只是入口，获批的参数草稿才是实际查库的口径，而解释是对实际结果的描述。如果解释写"6 月"而数据是 5 月的，结论和证据就对不上——下一章要讲的 FactRef 定位（点击结论数字跳到对应结果行）也会跟着失效。反过来，若工作台参数没有被批准修改，就继续采用原口径；不能假设月份一定能改。

&emsp;&emsp;这里还有一个常见误解："只改了一个日期，其他都没动。"实际上从参数到解释有一条完整的联动链，五个对象一起换新：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-7　参数改动后哪些东西会变</font></p>

<div class="center">

| 对象 | 参数改动后是否变化 | 判断作用 |
| :---: | :---: | :---: |
| `parameter_digest` | 变化 | 当前参数草稿和原冻结参数不同 |
| `plan_hash` | 变化 | 校验计划不能沿用到新草稿 |
| `validation_id` | 生成新值 | 新草稿取得新的校验记录 |
| `execution_id` | 生成新值 | 新校验绑定新执行 |
| 实际口径 | 跟随最终 execution | 解释必须按实际参数复述 |

</div>

&emsp;&emsp;这五行不是五件独立的事，而是同一条因果链的不同切面：`parameter_digest` 变了，说明草稿与冻结参数已经不同；`plan_hash` 跟着变，说明旧校验计划作废；于是校验记录换新的 `validation_id`，执行绑定新的 `execution_id`，解释的锚点也换成新口径。名字不用背，判断口径记一句：参数一变，从校验到执行到解释整条链全部换新。

&emsp;&emsp;还剩最后一个问题：这些结果在系统里到底算"多可信"？一次结果从产生到被采信，可信级别是逐级上升的，每一级都有明确的准入动作：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 6-8　可信级别怎样变化</font></p>

<div class="center">

| 阶段 | 可信级别 | 说明 |
| :---: | :---: | :---: |
| 冻结 Query / Seed | 待确认来源 | Seed 来自待确认的冻结 Query，验证阶段仍会执行 Workbench Policy，不认为 Seed 已自带校验通过 |
| 工作台执行 | `ANALYST_TEMPORARY` | 执行成功只是分析师临时结果，还不算分析已验证 |
| 受控回流 | `WORKBENCH_REVISION` + `ANALYSIS_VERIFIED` | 服务端复核状态、版本与血缘后，新快照才升为已验证 |
| Hermes 解读 | 引用 `ANALYSIS_VERIFIED` 快照 | 解释必须跟随回流后的实际口径 |

</div>

&emsp;&emsp;两句话收住判断边界：工作台执行成功只是 `ANALYST_TEMPORARY`（分析师临时结果），不等于 `ANALYSIS_VERIFIED`——只有受控回流、服务端复核后的新快照才是；Seed 也不是校验通过的凭证，验证阶段照样对它执行 Workbench Policy。可信级别只认动作、不认身份：没走完回流那一步，结果永远停留在"临时"。

### 6.5 先判断，再看参考答案

<details>
<summary>联动 SQL 已经 ALLOW，随后把获批月份从 6 月改成 5 月，此时可以直接执行吗？</summary>

不可以。旧 validation 与旧 execution 对当前参数草稿的提交资格已经失效；历史 execution 仍保留，但必须重新校验并生成与 5 月参数一致的新 execution。

</details>

&emsp;&emsp;**专项自测｜实际口径**：原问题写 2026 年 6 月，工作台若获批改成 2026 年 5 月，回流解释应该写哪个月份？答案是 2026 年 5 月；否则解释与实际执行不一致。

&emsp;&emsp;**本章自测**：你能按四段状态接力复述入口资格、参数草稿、重新校验与执行、受控回流，并解释为什么新参数必须对应新 execution、最终解释必须跟随实际口径。

&emsp;&emsp;**机制卡③　SQL 白名单与 plan_hash 冻结（两道关卡各自的底层）**：本章「自由 SQL 不能返回 Hermes」「参数变了必须重新校验」分别押在两道确定性判断上——前者缺的是本卡后半的血缘与冻结，后者缺的是本卡前半的哈希稳定性。`SqlPolicy.inspect`（[`nl2sql/policy/inspector.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/policy/inspector.py)，约 13 行起）只做四项判定——单条只读、对象边界、无星号、函数白名单，任何一条无法证明安全即拒绝（fail-closed）；`FrozenQueryFactory.freeze`（[`nl2sql/compilation/freeze.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/compilation/freeze.py)，约 24 行起）把规范化 SQL + 参数 + 完整计划 + 目录哈希 + 三份语义哈希整体打成 `plan_hash`。

&emsp;&emsp;下面直接运行 `SqlPolicy.inspect`，用正常 SQL 和四条恶意 SQL 验证白名单拦截：

In [17]:
# 机制卡③：SqlPolicy.inspect + FrozenQueryFactory.freeze 现场运行
from hermes_analytics.nl2sql.planning.planner import QueryPlanner
from hermes_analytics.nl2sql.compilation.compiler import SqlCompiler
from hermes_analytics.nl2sql.policy.inspector import SqlPolicy
from hermes_analytics.nl2sql.compilation.freeze import FrozenQueryFactory

# 先用合法 Intent 生成一份 plan（SqlPolicy 需要 plan 作为上下文）
intent = AnalysisIntent(
    metric="paid_orders",
    time_range=TimeRange(start=date(2026, 6, 1), end=date(2026, 6, 30)),
    time_grain=None, group_by=("channel",), filters=(),
    comparison="none", ranking=None,
)
plan = QueryPlanner().plan(intent, catalog)
compiled = SqlCompiler().compile(plan, catalog, catalog.content_hash)

# 正常 SQL：四项白名单全过
policy = SqlPolicy().inspect(compiled.sql, plan, catalog)
frozen = FrozenQueryFactory().freeze(compiled, plan, catalog)
print(f"正常SQL -> allowed={policy.allowed}")
print(f"  decisions={[d.rule_id for d in policy.decisions]}")
print(f"  plan_hash={frozen.plan_hash}")

# 四条恶意 SQL：全部被拦截
bad_sqls = [
    ("SELECT * FROM analytics.v_paid_order_items", "星号投影"),
    ("SELECT 1; DROP TABLE x", "多语句注入"),
    ("SELECT pg_sleep(10)", "禁止函数"),
    ("DELETE FROM analytics.v_paid_order_items", "非只读操作"),
]
for sql, desc in bad_sqls:
    p = SqlPolicy().inspect(sql, plan, catalog)
    print(f"{desc} -> allowed={p.allowed}")

正常SQL -> allowed=True
  decisions=['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS']
  plan_hash=sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83
星号投影 -> allowed=False
多语句注入 -> allowed=False
禁止函数 -> allowed=False
非只读操作 -> allowed=False


```text
正常SQL -> allowed=True
  decisions=['SQLP-001-SINGLE-READ', 'SQLP-002-GOVERNED-OBJECTS', 'SQLP-003-NO-STAR', 'SQLP-004-FUNCTIONS']
  plan_hash=sha256:d9e3342dee9e802b820332bb4c08ba8dd817b0e22f00206b4a83922155fb7a83
星号投影 -> allowed=False
多语句注入 -> allowed=False
禁止函数 -> allowed=False
非只读操作 -> allowed=False
```

&emsp;&emsp;先划清本卡讲的到底是哪道关卡：`SqlPolicy` 的四项判定（SQLP-001~004）是**第一道——安全闸门**，它在主链编译期运行，管"这条 SQL 危不危险"。工作台里自由 SQL 和联动 SQL 过的是**另一套独立白名单** `WorkbenchSqlPolicy`（[`application/workbench/policy.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/policy.py)，WBSQL-001~006）：同样有单条只读、治理对象、无星号、函数白名单四项，外加参数契约（WBSQL-005，命名参数必须真实出现在 SQL 里）和工作台预算（WBSQL-006，限制查询规模）两项。**自由 SQL 同样要过这套安检，一项不少**——它进不了回流，输的不是安检，是第二道关卡缺护照：没有 `source_query_id` 血缘，没有 plan_hash 冻结。

&emsp;&emsp;而 `freeze` 补的正是第二道关卡的凭证：`SqlCompiler.compile` 依视图契约拼出那条参数化 SQL（值只用命名占位符，不进 SQL 文本），`freeze` 把 SQL、参数、完整计划、领域包内容哈希和三份语义哈希（Intent、join、结果契约）整体打成 `plan_hash`。改计划任一处，`plan_hash` 就变——这正是参数变化后旧校验、旧执行资格必须作废的根因；因此即使 SQL 与参数一字未动，领域包一升级，旧冻结与旧校验同样作废。回看 6.1 的两道关卡：白名单判定（本卡前半）守住"能不能执行"，血缘与冻结（本卡后半）守住"能不能回流"——第六章的"受控"二字，由这两道关卡合写。

&emsp;&emsp;下一章问题：执行并返回分析工作区后，怎样证明结论中的一个数字来自这次真实结果？

## <center>第七章：结果与证据诊断</center>

&emsp;&emsp;前六章走完，页面上已经出现结论了。但“结论看起来合理”不等于“结论可信”——合理是措辞层面的事，可信需要证据。本章教你两套可直接上手的流程：**验证流程**（7.1：点一个数字，看它如何被定位到真实结果的行列——四步证据链）和**检查流程**（7.2：拿到任何数字先问三个问题——怎么跑出来的、按什么口径、在结果表哪里）。7.3 再补失败时分层诊断：查询失败、答案发布失败、证据校验失败各在哪一层、怎么分辨。本章核心问题：怎样证明一个数字来自当前这次真实执行？

> **证据定位状态**：FactRef 点击定位已有运行截图；解读失败界面尚未取得运行截图，以机制说明为主。

&emsp;&emsp;先拆误解：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 7-1　常见误解与真实机制</font></p>

<div class="center">

| 常见误解 | 真实机制 | 判断边界 |
| :---: | :---: | :---: |
| 结论文字合理就说明证据正确 | 结论数字必须指向同次 ResultSnapshot 的行、列和值 | 文字合理不能替代坐标核对 |
| FactRef 是模型自己编的编号 | FactRef 由服务端生成，模型只能从 allowed 集合选择 | 不在 allowed 集合就不能通过证据校验 |
| 结果已保存但解读失败等于查询失败 | Query、execution、Snapshot 与 Interpretation 分层 | 解读失败不直接重跑 SQL |
| 解读失败都可以在同一流程重试 | 只有解释器抛出特定 retryable `ApplicationError` 时，同一 operation 内最多重试一次 | 非可重试错误直接进入 Interpretation 失败 |

</div>

&emsp;&emsp;**源码锚点**：结果对象看 [`results.py`](HermesAnalytics第二版/backend/src/hermes_analytics/domain/results.py)，FactRef 生成看 [`evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py)，解释流程与特定错误的单次重试看 [`analysis_execution/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/service.py)，allowed FactRef 与答案发布看 [`publish_answer.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/tools/publish_answer.py)。

&emsp;&emsp;**口语类比与反例**：这像审计底稿——报告写得顺畅不代表数字有出处。反例是结论说“抖音最高”，FactRef 却指向另一行；或看到解读失败就直接重跑 SQL。

&emsp;&emsp;先看证据链全景——FactRef 从服务端生成到你点击高亮，中间经过四只手。这张表是全章的骨架，后面各节分别展开每一步：

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 7-2　FactRef 从生成到点击四步功能卡</font></p>

<div class="center">

| 步骤 | 谁做什么 | 源码锚点（当前版本约行段） | 一句话讲法 |
| :---: | :---: | :---: | :---: |
| 1 | EvidenceBuilder 从数值语义列生成 FactRef，绑定 query、snapshot、row、column、value | [`evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py) `EvidenceBuilder.build`，约 24–97；[`results.py`](HermesAnalytics第二版/backend/src/hermes_analytics/domain/results.py) `FactRef`，约 192–210 | 服务端把可引用数字做成证据坐标。 |
| 2 | `allowed_fact_ref_ids` 进入解释请求，模型只能从白名单选 | [`hermes_interpreter.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/hermes_interpreter.py) `OperationAnswerPublisher` / `HermesResultInterpreter`，约 35–97、169–260 | 模型拿到的白名单由服务端决定。 |
| 3 | 模型只提交 `fact_ref_ids`，不提交坐标 | [`evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py) `EvidenceBuilder.validate_answer`，约 98–180 | 模型不能自造引用或坐标。 |
| 4 | 前端点击按钮后定位并高亮相同 query/snapshot/row/column 的目标 | [`ConversationTurnCard.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/ConversationTurnCard.tsx) FactRef 按钮，约 41–180；[`AnalystConversationWorkspace.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx) `findFactReferenceTarget`，约 143–157；`focusFactReference`，约 814–820 | 点击后按坐标把结果表对应单元格高亮。 |

</div>

&emsp;&emsp;回流完成后的解读由 `AnalysisExecutionService._interpret_and_complete` 接管，当前版本约在 458–623。它把允许 FactRef、解读请求和结果发布串起来：模型提交答案，服务端校验 allowed 集合，通过后才算 Interpretation 完成。

### 7.1 一次完整的验证流程：从结论数字到高亮单元格

&emsp;&emsp;先站在学员视角把整件事走一遍。你在页面上看到模型给出结论：“抖音渠道净销售额最高，为 182035.93 元”。这句话可信吗？你可以立刻做一个动作——**点击结论里带下划线的“182035.93”**。页面跳到结果表，对应单元格被橙色高亮。这个动作背后是一条完整的证据链，本章把它拆成四步（表 7-2 就是这四步的源码索引）：

&emsp;&emsp;**生成**：SQL 执行成功后，服务端的 `EvidenceBuilder` 扫描结果里的数值列，把每个“可引用的数字”做成一个证据坐标——绑定它是哪个 Query 的哪次快照、第几行第几列、值是多少。这个坐标就叫 **FactRef**，承载它的结果快照叫 **ResultSnapshot**。关键在于：**坐标由服务端生成，不是模型编的**。

&emsp;&emsp;**约束**：模型解读结果时，解释请求里带着 allowed FactRef 白名单。模型写结论时只能引用白名单里的 ID，不能自造编号，也不能自己写“第 3 行第 2 列”这种坐标——它提交的只是一串 `fact_ref_ids`。

&emsp;&emsp;**校验**：模型答案提交后，服务端把答案里的 ID 与白名单做包含判定，越界引用直接判 `FACT_REF_INVALID`（机制卡④ 的三行代码就是这条判定）。所以你看到的每个可点击数字，都是**过了服务端证据校验的**。

&emsp;&emsp;**定位**：你点击数字时，前端按四重坐标（query_id + snapshot_id + row_key + column_key）在结果表里找唯一目标并高亮——注意是四重匹配，**另一个月快照里的相同数值不会被误高亮**。

&emsp;&emsp;四步连起来就是本章的验证答案：**一个数字可信，因为它能被定位到同一次真实执行的确切位置；定位链上没有任何一环由模型自由发挥。**下面的机制图表达页面展示层与服务端证据层的关系，不是运行截图。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172941483.png" alt="ResultSnapshot 与 FactRef 结果证据机制图" width="85%"></div>

&emsp;&emsp;机制图只说明证据结构；真实点击定位看下面的“运行截图｜Compose 项目 hermes-analytics-final”。点击结论中的“抖音 · 净销售额 182035.93”后，结果表对应 182035.93 单元格被橙色高亮。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172949717.png" alt="运行截图：点击抖音净销售额后结果表 182035.93 单元格橙色高亮" width="85%"></div>

&emsp;&emsp;补充一个名词边界：`snapshot_id` 是字段标识；课件把一次真实结果快照统称 ResultSnapshot 概念，但代码里不存在同名 Python dataclass——概念名，不是类名。

### 7.2 三种证据检查视角

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 7-3　三种证据检查视角的分工</font></p>

<div class="center">

| 检查视角 | 主要看什么 | 适合回答的问题 |
| :---: | :---: | :---: |
| 技术详情的执行状态 | Query、execution、实际参数与状态 | 执行到了哪一步？实际参数是什么？ |
| 技术详情中的 Intent / Context 页签 | 指标、维度与上下文口径 | 这个数字按什么口径计算？本轮继承了什么？ |
| FactRef | 结论数字对应的行、列和值 | 这个数字来自结果表哪里？ |

</div>

&emsp;&emsp;把三个视角变成你拿到任何结论数字后的**固定三连问**：

&emsp;&emsp;**第一问：这个数字怎么跑出来的？**——打开技术详情看执行状态。Query 到哪一步、execution 是否 SUCCEEDED、实际参数是什么（第六章 6.4 讲过：问的是 6 月、实际执行的可能是 5 月，这里就是核对处）。

&emsp;&emsp;**第二问：这个数字按什么口径算的？**——切到技术详情的 Intent / Context 页签。指标是哪个、按什么维度分组、本轮继承了上一轮什么条件（第五章的三层状态在这里可查）。

&emsp;&emsp;**第三问：这个数字在结果表哪里？**——用 FactRef 点击定位，验证它确实落在同一次 ResultSnapshot 的行、列上。

&emsp;&emsp;三问对应三种视角，顺序固定：先确认"跑过、且是这次跑的"，再确认"口径没偏"，最后确认"数字有出处"。任何一问对不上，就进入 7.3 的分层诊断。这三个视角都在会话的技术详情里，不暗示存在独立入口或页面。

### 7.3 查询与解读分层

&emsp;&emsp;三连问哪一问对不上，都需要同一套下探方法：把 Query、Execution、ResultSnapshot、Interpretation **四层分开看**。先立一个最容易混淆的区分：**“模型提交了答案”不等于“Interpretation 成功”**——模型可以提交结论文字，但 FactRef 未通过 allowed 集合校验时，Interpretation 仍是失败。页面上“可信结果已保存，但结果解读失败”这行提示说的就是这种状态：前三层都成功了，第四层挂了。

&emsp;&emsp;下面这张机制图把四层分开：Query、Execution、ResultSnapshot、Interpretation，并区分 A/B 两类失败类型。它不是运行截图。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260819172942886.png" alt="Query、Execution、ResultSnapshot、Interpretation 分层与 A/B 失败机制图" width="85%"></div>

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 7-4　两类 Interpretation 失败边界</font></p>

<div class="center">

| 失败类型 | 触发条件 | 重试边界 | 本次结果 |
| :---: | :---: | :---: | :---: |
| A：未发布可接受的结构化答案 | 解释器抛出错误，未发布可接受答案 | 最多两次调用：失败后最多额外重试 1 次；只捕获 retryable `ApplicationError` | 重试仍失败或错误不可重试时，Interpretation 失败 |
| B：`FACT_REF_INVALID` | 模型答案中的 `fact_ref_ids` 存在不属于本次 allowed 集合、未授权或无法验证的 ID | 位于重试循环之后且 retryable=false；不会自动回到同一流程修正 FactRef | 本次 Interpretation 失败，ResultSnapshot 保留用于诊断 |

</div>

&emsp;&emsp;A 与 B 都属于 Interpretation 失败，但发生位置不同：A 位于答案发布之前，Interpretation 重试最多两次调用（失败后最多额外重试 1 次），只捕获 retryable `ApplicationError`；B 位于模型提交答案之后、重试循环之外且 retryable=false。`FACT_REF_INVALID` 不会自动修正证据引用，也没有独立重新解读入口。解读失败不重跑 SQL。

&emsp;&emsp;固定诊断顺序：

```text
Query 状态
→ execution 状态
→ ResultSnapshot 是否存在
→ FactRef 是否对应行、列和值
→ interpretation 是否完成
```

&emsp;&emsp;页面显示失败时，该次 Interpretation 已经终止。此时只能按 Query → Execution → Snapshot → Interpretation 的顺序诊断，确认属于答案发布失败还是 `FACT_REF_INVALID`；不能写成“在同一流程继续处理”，也不直接重跑 SQL。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 7-5　问题出现在哪里、已成功到哪一步、先看哪个文件/页面</font></p>

<div class="center">

| 问题出现在哪里 | 已成功到哪一步 | 先看哪个文件/页面 | 一句话动作 |
| :---: | :---: | :---: | :---: |
| Query 未生成或状态不对 | 尚未进入执行 | 技术详情中的 Query 状态；`bridge.py` 规划边界 | 先看冻结与人工确认是否成立。 |
| execution 未成功 | Query 可以，但没取得结果 | 技术详情中的 execution 状态；`workbench/service.py` 执行 | 检查校验决定、参数和实际结果状态。 |
| ResultSnapshot 未生成 | execution 成功，但回流后未生成 `ANALYSIS_VERIFIED` 快照 | `analysis_execution_repository.py` `accept_workbench_revision` 回流接受条件 | 先检查回流接受条件是否全部满足，不能只看工作台临时结果。 |
| Interpretation 失败 | Query、execution、Snapshot 已成功 | FactRef 按钮与 allowed 集合；`hermes_interpreter.py` 发布 | 区分答案发布失败与 `FACT_REF_INVALID`，不重跑 SQL。 |

</div>

&emsp;&emsp;诊断时按 Query → Execution → Snapshot → Interpretation 的顺序逐层下探；看到失败界面先确认已完成到哪一层，再去看对应文件和页面，不要直接重跑数据库。

> **待运行验证｜失败界面**：`interpretation_failure_sample` 尚未取得真实运行截图，因此本段仍保留“机制/待运行验证”边界，不伪造失败截图。

### 7.4 本章自测

&emsp;&emsp;问题：页面提示“可信结果已保存，但结果解读失败”时，哪些步骤已经成功，下一步应先检查什么？参考答案：Query、execution 和 ResultSnapshot 可能已经成功；下一步区分“未发布可接受的结构化答案”与 `FACT_REF_INVALID`，并按终态失败诊断，不直接重跑 SQL。通过标准：你能选择正确的证据检查视角，并说明两类 Interpretation 失败的重试边界。

&emsp;&emsp;**机制卡④　证据引用只能选、不能造**：`EvidenceBuilder.validate_answer`（[`analysis_execution/evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py)，约 98 行起）把模型答案里的 `fact_ref_ids` 与本次执行生成的 allowed 集合做 `issubset` 判定；任何越界引用都抛 `FACT_REF_INVALID`，Interpretation 失败但 ResultSnapshot 保留。

&emsp;&emsp;源码里这条判定就是下面三行（讲机制用，不可独立运行）：

```text
requested = tuple(dict.fromkeys(answer.fact_ref_ids))
if not set(requested).issubset(allowed):
    raise ApplicationError(code="FACT_REF_INVALID", ...)
```

&emsp;&emsp;对照第 7.3 节的表 7-4：B 类失败（`FACT_REF_INVALID`）发生在模型提交答案之后、重试循环之外且 retryable=false。这印证了「结论数字必须指向白名单内真实坐标」——不是模型措辞合理就通过，而是坐标必须落在服务端生成的证据集合内。

&emsp;&emsp;下一章问题：面对一条新的连续分析任务，你能独立完成全部判断吗？

## <center>第八章：迁移任务与总结</center>

&emsp;&emsp;前七章我们已经建立了完整判断链。现在用一条新的半完成案例验收：面对相似任务时，你能不能独立完成澄清、上下文、SQL 模式、参数复核和证据定位。

### 8.1 迁移任务

半完成案例：

```text
第 1 轮：2026 年 6 月抖音渠道净销售额
第 2 轮：只看华东地区
第 3 轮：哪个渠道表现最好
第 4 轮：重新分析，2026 年 6 月华东地区各渠道支付订单数
当前状态：第 4 轮已冻结，等待用户确认
```

你需要补全六项判断：

1. 第 3 轮缺什么信息；
2. 第 3 轮哪些旧筛选应丢弃；
3. 第 4 轮是否继承第 2、3 轮上下文；
4. 哪种 SQL 模式可以返回 Hermes；
5. 修改日期后，执行前还缺哪一步；
6. 返回后如何证明一个数字来自真实结果。

&emsp;&emsp;先独立判断，再与同伴核对，最后对照六项标准检查。只使用第1-7章已经建立的状态、边界和源码锚点，不引入新功能或新错误码。

&emsp;&emsp;提示一个贯穿点：第 3 轮正是第五章 5.1 讲过的“抖音陷阱”场景——前两轮已把范围锁到抖音，这一轮却问全渠道比较。判断它时先确认分析对象是否变化，再判断旧筛选是否仍有语义；机械继承会把“哪个渠道最好”答成“抖音内部比高低”。

### 8.2 最小变体设计

&emsp;&emsp;在上面的案例基础上，自行替换一个指标、一个维度和一次上下文变化。

1. 写出首轮问题可能缺少的指标、时间或 Top N。
2. 写出第二轮应继承、追加或丢弃的条件。
3. 判断本次变化是否需要明确重置。
4. 选择自由 SQL 或会话联动 SQL，并说明参数变化后是否需要重新校验。
5. 写出回流后如何用 FactRef 证明一个结论数字。

&emsp;&emsp;互检边界：不替用户猜口径；不机械继承旧筛选；不把重置理解为删除历史；不让自由 SQL 返回 Hermes；不沿用旧 validation；不把模型文字当作 FactRef 证据。

### 8.3 七项学习成果

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 8-1　全课七项学习成果</font></p>

<div class="center">

| 序号 | 学习成果 | 通过标准 |
| :---: | :---: | :---: |
| 1 | 澄清模糊问题 | 不替用户猜指标或时间 |
| 2 | 判断语义边界 | 区分澄清、业务拒绝和系统故障 |
| 3 | 管理连续上下文 | 正确处理继承、追加、切换和重置 |
| 4 | 区分 SQL 模式 | 依据来源与血缘判断返回资格 |
| 5 | 判断受控复核链 | 能按验收卡复述改参数 → 重校验 → 新 execution → 回流 |
| 6 | 核对实际口径 | 最终结论跟随实际执行参数 |
| 7 | 定位与诊断证据 | 能按验收卡复述 FactRef 定位并分层判断失败 |

</div>

### 8.4 复述主任务

&emsp;&emsp;请用唯一主任务复述：从“哪个渠道表现最好”开始，先补净销售额与 2026 年 6 月；追问“只看华东地区呢”“支付订单数呢”；执行“重新分析”；提交“2026 年 6 月华东地区各渠道净销售额”；停在人工确认点；进入会话联动 SQL；在批准范围内修改参数、重新校验、执行；以 `execution_id + source_query_id` 返回分析工作区；核对实际执行口径；最后点击结论数字定位 FactRef。

### 8.5 源码索引

&emsp;&emsp;下面把源码导航升级为“直播源码导航索引”。函数/组件符号是稳定导航，行号是当前版本辅助定位；源码变化后优先搜索符号。

<style>
.center{width:auto;display:table;margin-left:auto;margin-right:auto;}
.table-caption{text-align:center!important;width:100%;}
</style>

<p class="table-caption"><font face="黑体" size=4>表 8-2　直播源码导航索引</font></p>

<div class="center">

| 功能 | 文件 | 精确符号 | 当前行段 | 重点看什么 | 能证明什么 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 澄清与缺口 | [`intent/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/intent/service.py) | `IntentValidator.validate`、`ClarificationDetector.find_gaps` | 约 95–230、246–296 | `INTENT_INCOMPLETE`、`gaps` | 缺口判定在确定性 Core，不由模型自由发挥 |
| 模型组织澄清 | [`hermes_adapter/prompt.py`](HermesAnalytics第二版/backend/src/hermes_analytics/hermes_adapter/prompt.py) | `PromptComposer.compose` | 约 341 起 | 结构化输出与澄清工具 | 中文问句由模型把原因码翻译成用户语言 |
| 规划 | [`planner.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/planning/planner.py) | `QueryPlanner.plan` | 约 40 行起 | Intent → QueryPlan | 完整意图确定性地转成可重放计划（可现场离线跑） |
| 编译 | [`compiler.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/compilation/compiler.py) | `SqlCompiler.compile` | 约 23–45 行 | 参数化 SQL | SQL 由确定性编译器按视图契约拼出，非模型写出（可现场离线跑） |
| 白名单 | [`inspector.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/policy/inspector.py) | `SqlPolicy.inspect` | 约 13 行起 | SQLP-001~004 | 单条只读/对象边界/无星号/函数白名单，fail-closed（可现场离线跑） |
| 冻结 | [`freeze.py`](HermesAnalytics第二版/backend/src/hermes_analytics/nl2sql/compilation/freeze.py) | `FrozenQueryFactory.freeze`、`FrozenSnapshotVerifier.verify` | 约 24–62、63 行起 | plan_hash + 四份哈希 | 内容寻址防篡改，改任一处哈希即变（可现场离线跑） |
| 规划与冻结 | [`analysis_planning/bridge.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_planning/bridge.py) | `validate_submission`、`plan_queries`、`compile_queries`、`inspect_policy`、`_finish_if_ready` | 约 253–412、709 起 | 校验、规划、编译、策略、冻结 | 完整 Intent 通过校验后，确定性规划到冻结 Query 才进入人工确认 |
| 页面澄清卡 | [`ConversationTurnCard.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/ConversationTurnCard.tsx) | `ConversationTurnCard` | 约 41–280 | 澄清展示、FactRef 按钮 | 页面是展示出口，不是判定来源 |
| 重置上下文 | [`AnalystConversationWorkspace.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx) | `resetContext` | 约 1527–1556 | 点击后调用网关重置 | “重新分析”从页面发起 |
| 重置存储 | [`control_repository.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/postgres/control_repository.py) | `reset_analysis_context` | 约 374–442 | 只清 `last_intent`/`last_query_id` | 重置不删 `conversation_turns` |
| 联动入口 | [`AnalystWorkspacePage.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/pages/AnalystWorkspacePage.tsx) | `openSql` | 约 66 起 | 从查询跳转 SQL 工作台 | 联动入口来自待确认 Query |
| 来源与参数 | [`queries.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/queries.py) | `QueryProjectionService.workbench_seed` | 约 38–67 | Seed 可用性 | Seed 只存在于待确认且未过期的 Query |
| 工作台契约 | [`workbench_repository.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/postgres/workbench_repository.py) | `get_seed` | 约 158–228 | `query.status`、`execution_gate`、`sql_snapshot`、`parameter_contract` | Seed 从待确认且未过期的冻结 Query 重新构造 |
| 参数约束 | [`workbench.py`](HermesAnalytics第二版/backend/src/hermes_analytics/api/schemas/workbench.py) | `WorkbenchValidationRequest.validate_variant` | 约 70–85 | 联动只允许 `parameter_values` | 浏览器不能提交 SQL/拼接参数 |
| 校验 | [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py) | `WorkbenchService.validate` | 约 58–168 | ALLOW/DENY、`parameter_digest`、`plan_hash` | 后端是最终校验裁决者 |
| 派生材料 | [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py) | `_derived_material` | 约 170–276 | 来源、参数、版本、`source_query_id` | 联动资格由服务端组装 |
| 参数边界 | [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py) | `_reject_values_outside_catalog`、`_reject_values_outside_data_coverage`、`_reject_relation_violations` | 约 280–306、309–342、344–374 | 枚举、日期覆盖、参数关系 | 非法参数在确定性层被拒绝 |
| 执行 | [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py) | `WorkbenchService.execute` | 约 376–523 | 校验未过期、plan_hash 一致、执行 | 新 validation 绑定新 execution |
| 自由 SQL 资格 | [`workbench/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/workbench/service.py) | `validate` 中 `return_eligible=False` | 约 58–168 | `source_query_id=None`、`return_eligible=False` | 自由 SQL 可执行，但不能返回 Hermes |
| 回流路由 | [`turns.py`](HermesAnalytics第二版/backend/src/hermes_analytics/api/routes/turns.py) | `resume_from_workbench` | 约 203–243 | `execution_id + source_query_id` | 浏览器只提交引用，服务端复核血缘 |
| 回流接受 | [`analysis_execution_repository.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/postgres/analysis_execution_repository.py) | `accept_workbench_revision` | 约 242–691 | 来源、execution、validation、result 血缘 | 全部条件匹配才生成回流 |
| 实际口径 | [`results.py`](HermesAnalytics第二版/backend/src/hermes_analytics/domain/results.py) | `executed_query_scope`、`scope_revision_notes` | 约 297–372 起 | 实际参数覆盖意图口径 | 解读必须跟随实际执行参数 |
| 证据生成 | [`evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py) | `EvidenceBuilder.build` | 约 24–97 | 数值语义列生成 FactRef | FactRef 由服务端生成 |
| FactRef 白名单 | [`hermes_interpreter.py`](HermesAnalytics第二版/backend/src/hermes_analytics/infrastructure/hermes_interpreter.py) | `OperationAnswerPublisher` | 约 35–97 | allowed FactRef、发布校验 | 模型只能从白名单选择 |
| 解读校验 | [`evidence.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/evidence.py) | `validate_answer` | 约 98–180 | `fact_ref_ids` 白名单 | 模型只提交 ID，不提交坐标 |
| 结果解读 | [`analysis_execution/service.py`](HermesAnalytics第二版/backend/src/hermes_analytics/application/analysis_execution/service.py) | `_interpret_and_complete` | 约 458–623 | 回流后解释、allowed 集合 | 解读成功必须经过证据校验 |
| 前端定位 | [`AnalystConversationWorkspace.tsx`](HermesAnalytics第二版/frontend/src/features/analysis/components/AnalystConversationWorkspace.tsx) | `findFactReferenceTarget`、`focusFactReference` | 约 143–157、814–820 | 按 query/snapshot/row/column 定位 | 点击后高亮真实结果单元格 |

</div>

&emsp;&emsp;索引说明：函数/组件符号是稳定导航，行号是当前版本辅助定位，源码变化后优先搜索符号，不要死记行号。

### 8.6 证据标签

　　`源码核验` 表示结论来自 `HermesAnalytics第二版` 源码；`机制示意` 表示这是机制图，用于说明组件关系与预期边界，不是运行截图；`运行截图` 表示该图为本地 Compose 项目的运行证据，不承诺普遍保证；`待运行验证` 表示机制与验收条件已有源码依据，但对应界面行为尚未取得当前运行证据。课件中的代码输出仅说明该单元按既定顺序在本地运行时观察到的结果，不替代完整服务栈验收。

### 8.7 最终自测

&emsp;&emsp;问题：面对一条新的连续分析任务，你能否依次回答：当前缺什么、哪些条件仍适用、是否需要重置、该走哪种 SQL、参数变化后缺哪一步、结论数字由什么证据证明？通过标准：你能交出表 8-1 的七项学习成果，并为每项给出对应判断顺序。

### 8.8 后续课程

&emsp;&emsp;本课全部围绕数据分析师连续分析主线展开。分析师看到的指标、维度、参数边界和分析方法由谁维护，又怎样安全变更并发布？数据库管理员主线和归因工作台在后续课程展开。归因工作台站在本节主线之上、只消费「已完成分析」作冻结证据（候选来自 `COMPLETED + fact_refs` 的轮次，`infrastructure/postgres/attribution_repository.py`）；A/B 主张方与质疑方同阶段并行、Judge 不按票数而按冻结证据裁决（`application/attribution/operation.py`）；每条引用必须落在冻结证据白名单内（`application/attribution/assembler.py`）——它仍是「可信」子集，不是另开一条越过模型的通道。